# Fundalux — train the screening models (Kaggle notebook)

This notebook trains the two models behind Fundalux and measures them against the screening targets
(sensitivity above 90% and specificity above 85% for referable DR on held-out photographs):

1. **Image-quality gate** — rejects photographs that cannot be graded.
2. **DR classifier** — an ensemble of three networks (EfficientNet-B3, EfficientNetV2-S, ConvNeXt-Tiny) for
   referable DR (moderate NPDR or worse) plus the 5-level grade, with Grad-CAM heatmaps.

**Before you run it** (right-hand sidebar):

1. *Session options → Accelerator →* **GPU T4 x2**
2. *Session options → Internet →* **On** (needed once, for the ImageNet starting weights)
3. *Input → Add Input* — add the four datasets listed in section 4. (If you skip this, the notebook attaches
   or downloads them itself.)

Then **Save Version → Save & Run All (Commit)**. The run continues on Kaggle's servers with the browser
closed, so there is nothing to disconnect from. Nothing has to be uploaded and no cloud drive is involved.
Expect several hours: three models are trained at up to 512 px. Training is cut short rather than run past
`TIME_BUDGET_HOURS`, to stay inside Kaggle's 12-hour limit.

**Result:** `fundalux_outputs.zip` in the notebook's **Output**. Download it and unzip it into the `models/`
folder of the project; `models/README.md` lists what goes where.

## 1. Settings

In [ ]:
import os

# ---- Things you may want to change -----------------------------------------
FRESH_START = False       # True = ignore saved checkpoints and train from scratch

IMG_SIZE = 512            # every image is cropped to the retina and cached at this size
MODELS = [                # the DR ensemble; "size" is the input size of that model, "batch" and "lr" are its own
    {"arch": "efficientnet_b3", "size": 512, "batch": 12, "lr": 2e-4},    # the model of the first run, now at 512 px
    {"arch": "efficientnet_v2_s", "size": 448, "batch": 12, "lr": 2e-4},  # 2021: faster-training EfficientNet
    {"arch": "convnext_tiny", "size": 448, "batch": 12, "lr": 1e-4},      # 2022: the newest design of the three
]
EPOCHS = 8
BALANCE_SOURCES = True    # sample every source dataset with the same share of referable cases (see section 7)
MAX_EYEPACS = None        # e.g. 8000 to use a random part of EyePACS for a quick first run
TIME_BUDGET_HOURS = 10.5  # Kaggle stops a session at 12 h; training is cut short rather than cross this
EXPORT_ONNX = True        # ONNX copies for the web app (it runs without PyTorch); doubles the size of the zip
# What the quality gate learns to call gradable. Run 2 (16 Nov 2026) used 0.05-0.3 / 0.7-1.0: photographs degraded
# to severity 0.3 were accepted while the classifier's specificity fell to 53-79%. From run 3 on, the gate learns to
# refuse from 0.3. (Chosen after seeing run 2's test results; the robustness section re-measures it.)
GATE_GRADABLE_DEGRADATION = (0.03, 0.15)
GATE_UNGRADABLE_DEGRADATION = (0.3, 1.0)

QUALITY_ARCH = "efficientnet_b0"
QUALITY_SIZE = 256
QUALITY_EPOCHS = 4
QUALITY_BATCH = 32
QUALITY_LR = 3e-4

TARGET_SENSITIVITY = 0.90  # screening targets on held-out data
TARGET_SPECIFICITY = 0.85
VAL_SENSITIVITY = 0.93     # the decision threshold is tuned to this on validation data (safety margin)
SEED = 42

DATASETS = {               # public Kaggle datasets: name -> "owner/dataset"
    "eyepacs": "tanlikesmath/diabetic-retinopathy-resized",
    "aptos": "mariaherrerot/aptos2019",
    "idrid": "aaryapatel98/indian-diabetic-retinopathy-image-dataset",
    "messidor": "mariaherrerot/messidor2preprocess",
}
LOCAL_DATA = {}            # optional, e.g. {"aptos": "/kaggle/input/my-aptos-upload"} to use a copy you uploaded yourself
EYEQ_URL = "https://raw.githubusercontent.com/HzFu/EyeQ/master/data/Label_EyeQ_train.csv"

# ---- Smoke test: a tiny synthetic run that only checks the code. Leave it alone. ----
SMOKE_TEST = os.environ.get("FUNDALUX_SMOKE") == "1"
if SMOKE_TEST:
    IMG_SIZE, EPOCHS = 96, 2
    MODELS = [{"arch": "efficientnet_b0", "size": 96, "batch": 8, "lr": 3e-4}, {"arch": "efficientnet_v2_s", "size": 64, "batch": 8, "lr": 3e-4}, {"arch": "convnext_tiny", "size": 64, "batch": 8, "lr": 1e-4}]
    QUALITY_SIZE, QUALITY_EPOCHS, QUALITY_BATCH = 64, 1, 8

In [ ]:
import gc
import json
import math
import shutil
import socket
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

RUN_START = time.time()
IN_KAGGLE = os.path.isdir("/kaggle/working")
IN_COLAB = "google.colab" in sys.modules
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print(f"torch {torch.__version__} | device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if USE_AMP else ""))


def online(host="download.pytorch.org"):
    try:
        socket.create_connection((host, 443), timeout=8).close()
        return True
    except OSError:
        return False


if not SMOKE_TEST:
    if not USE_AMP:
        raise RuntimeError("No GPU found. Kaggle: Session options > Accelerator > GPU T4 x2. (Colab: Runtime > Change runtime type > T4 GPU.) Then run everything again.")
    if not online():
        raise RuntimeError("No internet access. Kaggle: Session options > Internet > On (your account must be phone-verified). It is needed for the ImageNet starting weights.")

# WORK is what you get back at the end; SCRATCH holds the big temporary files (on Kaggle it is not saved).
WORK = Path("/kaggle/working") if IN_KAGGLE else Path("/content") if IN_COLAB else Path.cwd()
SCRATCH = Path("/kaggle/temp") if IN_KAGGLE else WORK
SCRATCH.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
os.makedirs("fundalux", exist_ok=True)

cv2.setNumThreads(0)  # OpenCV threads and forked DataLoader workers do not mix
torch.manual_seed(SEED)
np.random.seed(SEED)

## 2. Project code

The next cells write the `fundalux` Python package next to this notebook. It is the same code the local app
uses, so an image is preprocessed in exactly the same way during training and at screening time.
(These cells are generated from the project's `fundalux/` folder — edit the files there, not here.
If you do change one of these cells, restart the session before running the rest.)

In [ ]:
%%writefile fundalux/__init__.py
"""Fundalux: explainable referable diabetic-retinopathy screening.

The package is shared by the training notebook and the local app so that
training and inference run exactly the same preprocessing.
"""

__version__ = "0.1.0"

# International Clinical DR scale, as used by APTOS / EyePACS / IDRiD / Messidor-2.
GRADE_NAMES = ["No DR", "Mild NPDR", "Moderate NPDR", "Severe NPDR", "Proliferative DR"]

# Referable DR = moderate NPDR or worse. (Diabetic macular edema is also
# referable clinically, but the training sets carry no DME labels.)
REFERABLE_MIN_GRADE = 2

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

In [ ]:
%%writefile fundalux/preprocess.py
"""Fundus preprocessing: crop to the retina, illumination correction, CLAHE.

All functions take and return RGB uint8 arrays.
"""

from __future__ import annotations

import functools
from dataclasses import dataclass

import cv2
import numpy as np


@dataclass(frozen=True)
class CropGeometry:
    """Square region of the source image that holds the fundus disc."""

    x0: int
    y0: int
    side: int
    found: bool  # False when no fundus disc was detected (centre crop used)


def decode_image(data: bytes) -> np.ndarray:
    """Decode encoded image bytes (JPEG/PNG/TIFF...) to RGB uint8."""
    img = cv2.imdecode(np.frombuffer(data, dtype=np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        import io

        from PIL import Image

        return np.asarray(Image.open(io.BytesIO(data)).convert("RGB"))
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def read_image(path) -> np.ndarray:
    with open(path, "rb") as f:
        return decode_image(f.read())


def channel_max(img: np.ndarray) -> np.ndarray:
    """Per-pixel maximum over R, G, B (much faster than ndarray.max(axis=2))."""
    r, g, b = cv2.split(img)
    return cv2.max(cv2.max(r, g), b)


def find_fundus(img: np.ndarray) -> CropGeometry:
    """Locate the fundus disc and return the square that encloses it.

    The disc is often cut off at the top and bottom of the frame, so the square
    side is the larger bounding-box dimension and may extend past the image.
    """
    h, w = img.shape[:2]
    scale = min(1.0, 256.0 / max(h, w))
    small = img
    if scale < 1.0:
        small = cv2.resize(img, (max(1, round(w * scale)), max(1, round(h * scale))), interpolation=cv2.INTER_AREA)
    gray = channel_max(small)
    thr = max(10.0, 0.1 * float(np.percentile(gray, 95)))
    mask = (gray > thr).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n > 1:
        i = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        x, y, bw, bh, area = (int(v) for v in stats[i])
        if area >= 0.08 * mask.size and min(bw, bh) >= 0.3 * max(bw, bh):
            side = max(bw, bh) / scale
            cx, cy = (x + bw / 2) / scale, (y + bh / 2) / scale
            return CropGeometry(round(cx - side / 2), round(cy - side / 2), max(1, round(side)), True)
    side = min(h, w)
    return CropGeometry((w - side) // 2, (h - side) // 2, side, False)


def apply_crop(img: np.ndarray, geom: CropGeometry, size: int, interpolation: int | None = None) -> np.ndarray:
    """Cut the square described by `geom` (zero-padded if needed) and resize it."""
    h, w = img.shape[:2]
    x0, y0, x1, y1 = geom.x0, geom.y0, geom.x0 + geom.side, geom.y0 + geom.side
    crop = img[max(0, y0) : min(h, y1), max(0, x0) : min(w, x1)]
    pad = (max(0, -y0), max(0, y1 - h), max(0, -x0), max(0, x1 - w))
    if any(pad):
        crop = cv2.copyMakeBorder(crop, *pad, cv2.BORDER_CONSTANT, value=0)
    if interpolation is None:
        interpolation = cv2.INTER_AREA if geom.side > size else cv2.INTER_LINEAR
    return cv2.resize(crop, (size, size), interpolation=interpolation)


@functools.lru_cache(maxsize=8)
def _disc_u8(size: int) -> np.ndarray:
    c = (size - 1) / 2
    yy, xx = np.ogrid[:size, :size]
    mask = ((xx - c) ** 2 + (yy - c) ** 2 <= (0.985 * size / 2) ** 2).astype(np.uint8) * 255
    mask.setflags(write=False)
    return mask


def disc_mask(size: int) -> np.ndarray:
    """Boolean mask of the inscribed fundus disc for a `size` x `size` crop."""
    return _disc_u8(size) > 0


def keep_where(img: np.ndarray, mask_u8: np.ndarray) -> np.ndarray:
    """Copy of `img` with every pixel outside the 0/255 mask set to black."""
    return cv2.bitwise_and(img, img, mask=mask_u8)


def crop_fundus(img: np.ndarray, size: int) -> tuple[np.ndarray, CropGeometry]:
    """Crop to the fundus disc, resize to `size` and black out everything outside it."""
    geom = find_fundus(img)
    return keep_where(apply_crop(img, geom, size), _disc_u8(size)), geom


def retina_mask(img: np.ndarray) -> np.ndarray:
    """0/255 mask of disc pixels that actually contain retina (the disc may be cut off)."""
    lit = cv2.threshold(channel_max(img), 8, 255, cv2.THRESH_BINARY)[1]
    return cv2.bitwise_and(lit, _disc_u8(img.shape[0]))


def _flatten_lightness(L: np.ndarray, valid: np.ndarray, target: float) -> np.ndarray:
    size = L.shape[0]
    Lf = L.astype(np.float32)
    v = valid.astype(np.float32) / 255.0
    # The illumination field is smooth, so estimate it on a small copy.
    s = min(size, 96)
    Ls = cv2.resize(Lf * v, (s, s), interpolation=cv2.INTER_AREA)
    vs = cv2.resize(v, (s, s), interpolation=cv2.INTER_AREA)
    sigma = 0.12 * s
    field = cv2.GaussianBlur(Ls, (0, 0), sigma) / np.maximum(cv2.GaussianBlur(vs, (0, 0), sigma), 1e-3)
    field = cv2.resize(field, (size, size), interpolation=cv2.INTER_LINEAR)
    gain = np.clip(target / np.maximum(field, 1.0), 0.25, 4.0)
    return np.clip(Lf * gain, 0, 255).astype(np.uint8)


def _clahe(L: np.ndarray, clip_limit: float, tile_grid: int) -> np.ndarray:
    return cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_grid, tile_grid)).apply(L)


def _on_lightness(img: np.ndarray, fn) -> np.ndarray:
    valid = retina_mask(img)
    if cv2.countNonZero(valid) < 0.01 * valid.size:
        return img.copy()
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    lab[..., 0] = fn(np.ascontiguousarray(lab[..., 0]), valid)
    return keep_where(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB), valid)


def correct_illumination(img: np.ndarray, target: float = 120.0) -> np.ndarray:
    """Flatten uneven illumination on a cropped fundus image.

    Lightness is divided by its own large-scale blur (the illumination field),
    so dark rims, vignetting and under-exposure are evened out while colour is kept.
    """
    return _on_lightness(img, lambda L, valid: _flatten_lightness(L, valid, target))


def apply_clahe(img: np.ndarray, clip_limit: float = 2.0, tile_grid: int = 8) -> np.ndarray:
    """Contrast-limited adaptive histogram equalisation on the lightness channel."""
    return _on_lightness(img, lambda L, valid: _clahe(L, clip_limit, tile_grid))


def enhance_fundus(cropped: np.ndarray, target: float = 120.0, clip_limit: float = 2.0, tile_grid: int = 8) -> np.ndarray:
    """Illumination correction followed by CLAHE, on an image from `crop_fundus`."""
    return _on_lightness(cropped, lambda L, valid: _clahe(_flatten_lightness(L, valid, target), clip_limit, tile_grid))


def model_input(cropped: np.ndarray, size: int) -> np.ndarray:
    """Classifier input from a cropped image: resize to the model's own input size, then enhance.

    Same order as training (the cached crop is resized first, enhanced last).
    """
    if cropped.shape[0] != size or cropped.shape[1] != size:
        cropped = cv2.resize(cropped, (size, size), interpolation=cv2.INTER_AREA)
    return enhance_fundus(cropped)


def preprocess_fundus(img: np.ndarray, size: int) -> tuple[np.ndarray, np.ndarray, CropGeometry]:
    """Full pipeline on a raw photograph. Returns (cropped, enhanced, geometry).

    `cropped` feeds the quality gate, `enhanced` feeds the DR classifier.
    """
    cropped, geom = crop_fundus(img, size)
    return cropped, enhance_fundus(cropped), geom

In [ ]:
%%writefile fundalux/degrade.py
"""Field-condition degradations for portable fundus cameras.

Used three ways: mild versions as training augmentation for the DR classifier,
severe versions as synthetic "ungradable" examples for the quality gate, and
fixed-severity sweeps to measure robustness.

Every function takes a cropped RGB uint8 fundus image (before enhancement) and a
`severity` in [0, 1]; sizes are relative to the image so behaviour is the same
at any resolution.
"""

from __future__ import annotations

import cv2
import numpy as np

from .preprocess import channel_max, keep_where

KINDS = ("blur", "motion", "glare", "haze", "dark")

# Severity ranges. Mild images stay gradable; severe ones are treated as
# ungradable. The gap in between is deliberately never generated for training.
MILD = (0.05, 0.3)
SEVERE = (0.7, 1.0)


def _blur(img, severity, rng):
    return cv2.GaussianBlur(img, (0, 0), max(0.03 * severity**1.5 * img.shape[0], 0.1))


def _motion(img, severity, rng):
    length = max(3, int(round(0.12 * severity**1.5 * img.shape[0])) | 1)
    kernel = np.zeros((length, length), np.float32)
    kernel[length // 2, :] = 1.0
    rot = cv2.getRotationMatrix2D(((length - 1) / 2, (length - 1) / 2), float(rng.uniform(0, 180)), 1.0)
    kernel = cv2.warpAffine(kernel, rot, (length, length))
    kernel /= max(float(kernel.sum()), 1e-6)
    return cv2.filter2D(img, -1, kernel)


def _glare(img, severity, rng):
    radius = 0.08 + 0.42 * severity  # as a fraction of the image width
    r, theta = 0.35 * np.sqrt(rng.uniform()), rng.uniform(0, 2 * np.pi)
    cx, cy = 0.5 + r * np.cos(theta), 0.5 + r * np.sin(theta)
    grid = (np.arange(64, dtype=np.float32) + 0.5) / 64  # the blob is smooth: build it small
    blob = np.exp(-((grid[None, :] - cx) ** 2 + (grid[:, None] - cy) ** 2) / (2 * radius**2))
    strength = 0.3 + 1.2 * severity
    veil = np.clip(255.0 * strength * cv2.resize(blob, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_LINEAR), 0, 255).astype(np.uint8)
    return cv2.add(img, cv2.merge([veil, veil, veil]))


def _haze(img, severity, rng):
    # Media opacity / dirty lens: light scatters, contrast collapses towards a bright veil.
    keep = 1.0 - 0.95 * severity
    soft = cv2.GaussianBlur(img, (0, 0), max(0.012 * severity * img.shape[0], 0.1))
    veil = np.empty_like(img)
    veil[:] = (200, 185, 170)
    return cv2.addWeighted(soft, keep, veil, 1.0 - keep, 0.0)


def _dark(img, severity, rng):
    # Under-exposure: signal shrinks while sensor noise does not.
    noise = rng.standard_normal(img.shape[:2], dtype=np.float32) * (1.0 + 3.0 * severity)
    out = img.astype(np.float32) * (0.02**severity) + noise[..., None]
    return np.clip(out, 0, 255).astype(np.uint8)


_FUNCS = {"blur": _blur, "motion": _motion, "glare": _glare, "haze": _haze, "dark": _dark}


def degrade(img: np.ndarray, kind: str, severity: float, rng: np.random.Generator | None = None) -> np.ndarray:
    """Apply one degradation. The black area outside the fundus stays black."""
    if severity <= 0:
        return img.copy()
    rng = rng or np.random.default_rng(0)
    out = _FUNCS[kind](img, float(min(severity, 1.0)), rng)
    return keep_where(out, cv2.threshold(channel_max(img), 8, 255, cv2.THRESH_BINARY)[1])


def random_degradation(img: np.ndarray, rng: np.random.Generator, severity_range=MILD, kinds=KINDS):
    """Apply a randomly chosen degradation. Returns (image, kind, severity)."""
    kind = kinds[int(rng.integers(len(kinds)))]
    severity = float(rng.uniform(*severity_range))
    return degrade(img, kind, severity, rng), kind, severity

In [ ]:
%%writefile fundalux/quality.py
"""Image-quality gate helpers: cheap rule checks and human-readable hints.

The learned part of the gate is a FundusNet with a single "ungradable" logit;
this module holds the parts that need no training.
"""

from __future__ import annotations

import cv2
import numpy as np

from .preprocess import CropGeometry, disc_mask, retina_mask

_STATS_SIZE = 256


def quality_stats(cropped: np.ndarray) -> dict:
    """Simple image statistics of a cropped fundus image, at a fixed scale."""
    small = cv2.resize(cropped, (_STATS_SIZE, _STATS_SIZE), interpolation=cv2.INTER_AREA)
    disc = disc_mask(_STATS_SIZE)
    valid = retina_mask(small) > 0
    stats = {"retina_fraction": float(valid.sum() / disc.sum())}
    if valid.sum() < 50:
        stats.update(brightness=0.0, contrast=0.0, sharpness=0.0, saturated_fraction=0.0, red_minus_blue=0.0)
        return stats
    gray = cv2.cvtColor(small, cv2.COLOR_RGB2GRAY).astype(np.float32)
    # Stay away from the disc edge, whose hard step would dominate the Laplacian.
    inner = cv2.erode(valid.astype(np.uint8), np.ones((9, 9), np.uint8)) > 0
    lap = cv2.Laplacian(gray, cv2.CV_32F)
    means = small[valid].mean(axis=0)
    stats.update(
        brightness=float(gray[valid].mean()),
        contrast=float(gray[valid].std()),
        sharpness=float(lap[inner].var()) if inner.any() else 0.0,
        saturated_fraction=float((gray[valid] >= 250).mean()),
        red_minus_blue=float(means[0] - means[2]),
    )
    return stats


def basic_checks(image: np.ndarray, geom: CropGeometry, stats: dict, min_side: int = 224) -> list[str]:
    """Hard failures that need no model. Returns a list of reasons (empty = pass)."""
    reasons = []
    if min(image.shape[:2]) < min_side:
        reasons.append(f"Image is too small ({image.shape[1]}x{image.shape[0]} px).")
    if not geom.found:
        reasons.append("No retina found - the image is too dark or is not a fundus photograph.")
    elif stats["retina_fraction"] < 0.5:
        reasons.append("Less than half of the retina is visible.")
    elif stats["red_minus_blue"] <= 0:
        reasons.append("This does not look like a colour fundus photograph.")
    return reasons


def quality_hints(stats: dict, ranges: dict | None) -> list[str]:
    """Explain a rejection by comparing statistics with those of good images.

    `ranges` maps a statistic to the [low, high] interval that covers nearly all
    good-quality training images (measured in the training notebook).
    """
    if not ranges:
        return []
    hints = []

    def below(key):
        return key in ranges and stats[key] < ranges[key][0]

    def above(key):
        return key in ranges and stats[key] > ranges[key][1]

    if below("sharpness"):
        hints.append("Looks blurred - refocus and hold the camera steady.")
    if below("brightness"):
        hints.append("Too dark - check pupil alignment and flash/illumination.")
    if above("brightness") or above("saturated_fraction"):
        hints.append("Over-exposed or glare - reduce illumination and avoid reflections.")
    if below("contrast"):
        hints.append("Low contrast - clean the lens; a cataract or other media opacity can also cause this.")
    return hints

In [ ]:
%%writefile fundalux/metrics.py
"""Screening metrics in plain numpy (no scikit-learn needed at inference time)."""

from __future__ import annotations

import numpy as np


def roc_curve(y_true, y_score) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return (fpr, tpr, thresholds); a sample is positive when score >= threshold."""
    y_true = np.asarray(y_true).astype(bool)
    y_score = np.asarray(y_score, dtype=np.float64)
    order = np.argsort(-y_score, kind="mergesort")
    y_true, y_score = y_true[order], y_score[order]
    last = np.r_[np.where(np.diff(y_score))[0], y_true.size - 1]  # last index of each distinct score
    tps = np.cumsum(y_true)[last]
    fps = (last + 1) - tps
    n_pos, n_neg = int(y_true.sum()), int((~y_true).sum())
    tpr = np.r_[0.0, tps / n_pos] if n_pos else np.full(last.size + 1, np.nan)
    fpr = np.r_[0.0, fps / n_neg] if n_neg else np.full(last.size + 1, np.nan)
    return fpr, tpr, np.r_[np.inf, y_score[last]]


def roc_auc(y_true, y_score) -> float:
    """Area under the ROC curve; NaN when only one class is present."""
    y_true = np.asarray(y_true).astype(bool)
    if y_true.all() or not y_true.any():
        return float("nan")
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return float(np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2))


def sens_spec(y_true, y_score, threshold: float) -> tuple[float, float]:
    """Sensitivity and specificity when predicting positive for score >= threshold."""
    y_true = np.asarray(y_true).astype(bool)
    pred = np.asarray(y_score) >= threshold
    n_pos, n_neg = int(y_true.sum()), int((~y_true).sum())
    sens = float((pred & y_true).sum() / n_pos) if n_pos else float("nan")
    spec = float((~pred & ~y_true).sum() / n_neg) if n_neg else float("nan")
    return sens, spec


def threshold_for_sensitivity(y_true, y_score, target: float) -> float:
    """Highest threshold whose sensitivity is at least `target` (best specificity)."""
    y_true = np.asarray(y_true).astype(bool)
    if not y_true.any():
        return 0.5
    _, tpr, thresholds = roc_curve(y_true, y_score)
    ok = np.where(tpr >= target)[0]
    return float(thresholds[ok[0]])


def youden_threshold(y_true, y_score) -> float:
    """Threshold that maximises sensitivity + specificity - 1."""
    y_true = np.asarray(y_true).astype(bool)
    if y_true.all() or not y_true.any():
        return 0.5
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    return float(thresholds[1:][int(np.argmax((tpr - fpr)[1:]))])


def bootstrap_ci(fn, y_true, y_score, n_boot: int = 1000, seed: int = 0, alpha: float = 0.05) -> tuple[float, float]:
    """Percentile bootstrap CI for fn(y_true, y_score). NaN resamples are ignored."""
    y_true, y_score = np.asarray(y_true), np.asarray(y_score)
    rng = np.random.default_rng(seed)
    n = y_true.size
    if n == 0:
        return float("nan"), float("nan")
    vals = np.array([fn(y_true[i], y_score[i]) for i in rng.integers(0, n, size=(n_boot, n))], dtype=np.float64)
    vals = vals[~np.isnan(vals)]
    if vals.size == 0:
        return float("nan"), float("nan")
    return float(np.quantile(vals, alpha / 2)), float(np.quantile(vals, 1 - alpha / 2))


def confusion_matrix(y_true, y_pred, n_classes: int) -> np.ndarray:
    """Rows = true class, columns = predicted class."""
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    np.add.at(cm, (np.asarray(y_true, dtype=int), np.asarray(y_pred, dtype=int)), 1)
    return cm


def quadratic_weighted_kappa(y_true, y_pred, n_classes: int) -> float:
    """Agreement on an ordinal scale; 1 = perfect, 0 = chance."""
    cm = confusion_matrix(y_true, y_pred, n_classes).astype(np.float64)
    n = cm.sum()
    if n == 0:
        return float("nan")
    idx = np.arange(n_classes)
    w = (idx[:, None] - idx[None, :]) ** 2 / (n_classes - 1) ** 2
    expected = np.outer(cm.sum(1), cm.sum(0)) / n
    denom = (w * expected).sum()
    return float(1.0 - (w * cm).sum() / denom) if denom > 0 else float("nan")


def screening_report(y_true, y_score, threshold: float, n_boot: int = 1000, seed: int = 0) -> dict:
    """AUC, sensitivity and specificity at `threshold`, each with a 95% bootstrap CI."""
    y_true, y_score = np.asarray(y_true).astype(int), np.asarray(y_score, dtype=np.float64)
    sens, spec = sens_spec(y_true, y_score, threshold)
    report = {
        "n": int(y_true.size),
        "n_positive": int(y_true.sum()),
        "threshold": float(threshold),
        "auc": roc_auc(y_true, y_score),
        "sensitivity": sens,
        "specificity": spec,
    }
    if n_boot:
        report["auc_ci"] = bootstrap_ci(roc_auc, y_true, y_score, n_boot, seed)
        report["sensitivity_ci"] = bootstrap_ci(lambda t, s: sens_spec(t, s, threshold)[0], y_true, y_score, n_boot, seed)
        report["specificity_ci"] = bootstrap_ci(lambda t, s: sens_spec(t, s, threshold)[1], y_true, y_score, n_boot, seed)
    return report


def crossfit_operating_point(y_true, y_score, target: float, groups=None, repeats: int = 20, seed: int = 0) -> dict:
    """Honest estimate of what a threshold tuned for `target` sensitivity delivers on new data.

    Repeatedly splits the data in two halves (whole groups at a time, e.g. both
    eyes of a patient), tunes the threshold on one half, applies it to the other,
    and swaps. Returns the mean and range over repeats of the pooled
    sensitivity and specificity.
    """
    y_true, y_score = np.asarray(y_true).astype(bool), np.asarray(y_score, dtype=np.float64)
    groups = np.arange(y_true.size) if groups is None else np.asarray(groups)
    names = np.unique(groups)
    rng = np.random.default_rng(seed)
    sens, spec = [], []
    for _ in range(repeats):
        half = np.isin(groups, rng.permutation(names)[: names.size // 2])
        predicted = np.zeros(y_true.size, dtype=bool)
        for fit, apply in ((half, ~half), (~half, half)):
            predicted[apply] = y_score[apply] >= threshold_for_sensitivity(y_true[fit], y_score[fit], target)
        sens.append(float((predicted & y_true).sum() / max(y_true.sum(), 1)))
        spec.append(float((~predicted & ~y_true).sum() / max((~y_true).sum(), 1)))
    return {
        "sensitivity": float(np.mean(sens)), "specificity": float(np.mean(spec)),
        "sensitivity_range": (float(np.min(sens)), float(np.max(sens))), "specificity_range": (float(np.min(spec)), float(np.max(spec))),
        "repeats": repeats,
    }


def crossfit_referral_rate(y_true, y_score, target: float, groups=None, repeats: int = 20, seed: int = 0) -> np.ndarray:
    """For each image, the share of cross-fitting repeats in which it was referred.

    Same procedure as crossfit_operating_point, kept per image so results can be
    broken down (e.g. by DR grade) without ever using an image to tune its own threshold.
    """
    y_true, y_score = np.asarray(y_true).astype(bool), np.asarray(y_score, dtype=np.float64)
    groups = np.arange(y_true.size) if groups is None else np.asarray(groups)
    names = np.unique(groups)
    rng = np.random.default_rng(seed)
    referred = np.zeros(y_true.size)
    for _ in range(repeats):
        half = np.isin(groups, rng.permutation(names)[: names.size // 2])
        for fit, apply in ((half, ~half), (~half, half)):
            referred[apply] += y_score[apply] >= threshold_for_sensitivity(y_true[fit], y_score[fit], target)
    return referred / repeats

In [ ]:
%%writefile fundalux/models.py
"""Transfer-learning backbones with a small screening head."""

from __future__ import annotations

import numpy as np
import torch
from torch import nn

from . import IMAGENET_MEAN, IMAGENET_STD
from .inference import ensemble_probabilities  # noqa: F401  (re-exported)


class FundusNet(nn.Module):
    """ImageNet backbone + one linear head.

    Output column 0 is a binary logit (referable DR, or "ungradable" for the
    quality gate). Columns 1.. are DR-grade logits when `num_grades` > 0.
    """

    def __init__(self, arch: str = "efficientnet_b3", num_grades: int = 5, pretrained: bool = False, dropout: float = 0.3):
        super().__init__()
        import torchvision.models as tvm

        if not hasattr(tvm, arch):
            raise ValueError(f"Unknown architecture: {arch}")
        net = getattr(tvm, arch)(weights="DEFAULT" if pretrained else None)
        norm = nn.Identity()
        if arch.startswith("efficientnet"):
            self.features = net.features
            feat_dim = net.classifier[-1].in_features
        elif arch.startswith("convnext"):
            self.features = net.features
            norm = net.classifier[0]  # ConvNeXt normalises the pooled features before its classifier
            feat_dim = net.classifier[-1].in_features
        elif arch.startswith(("resnet", "resnext", "wide_resnet")):
            self.features = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool, net.layer1, net.layer2, net.layer3, net.layer4)
            feat_dim = net.fc.in_features
        else:
            raise ValueError(f"Unsupported architecture family: {arch} (use efficientnet_*, convnext_* or resnet*)")
        self.arch = arch
        self.num_grades = num_grades
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.norm = norm
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(feat_dim, 1 + num_grades)
        # Normalisation is part of the model, so every caller (training, the app,
        # the ONNX export) feeds plain 0-255 RGB and cannot get it wrong.
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1), persistent=False)

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        """`x` is a batch of RGB images with values 0-255 (uint8 or float), NCHW."""
        return self.features((x.float() / 255.0 - self.mean) / self.std)

    def forward_head(self, feats: torch.Tensor) -> torch.Tensor:
        return self.head(self.dropout(self.norm(self.pool(feats)).flatten(1)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.forward_head(self.forward_features(x))


def to_tensor(img: np.ndarray) -> torch.Tensor:
    """RGB uint8 HxWx3 -> uint8 CHW tensor (FundusNet normalises internally)."""
    return torch.from_numpy(np.ascontiguousarray(img.transpose(2, 0, 1)))


def save_weights(model: nn.Module, path) -> None:
    """Save a plain state_dict (float32, CPU) so it loads on any torch version."""
    state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    torch.save(state, str(path))


def load_weights(model: nn.Module, path) -> nn.Module:
    try:
        state = torch.load(str(path), map_location="cpu", weights_only=True)
    except TypeError:  # torch < 1.13 has no weights_only
        state = torch.load(str(path), map_location="cpu")
    model.load_state_dict(state)
    return model.eval()

In [ ]:
%%writefile fundalux/gradcam.py
"""Grad-CAM for FundusNet, plus overlap scores against lesion masks."""

from __future__ import annotations

import cv2
import numpy as np
import torch

from .heatmap import overlay_heatmap, resize_cam  # noqa: F401  (re-exported for the notebook)
from .models import FundusNet, to_tensor
from .preprocess import model_input


def grad_cam(model: FundusNet, x: torch.Tensor, target: int = 0) -> tuple[np.ndarray, torch.Tensor]:
    """Grad-CAM of output column `target` w.r.t. the last conv feature map.

    Returns (cams, outputs): cams is float32 [B, h, w], each scaled to [0, 1]
    (all zeros when nothing pushes the output up); outputs are the raw logits.
    """
    was_training = model.training
    model.eval()
    with torch.no_grad():
        feats = model.forward_features(x)
    with torch.enable_grad():
        feats = feats.detach().requires_grad_(True)
        out = model.forward_head(feats)
        grads = torch.autograd.grad(out[:, target].sum(), feats)[0]
    weights = grads.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((weights * feats).sum(dim=1)).detach().float()
    peak = cam.flatten(1).max(dim=1).values.clamp_min(1e-12).view(-1, 1, 1)
    cam = torch.where(peak > 1e-12, cam / peak, torch.zeros_like(cam))
    model.train(was_training)
    return cam.cpu().numpy(), out.detach()


def ensemble_grad_cam(members, cropped: list[np.ndarray], out_size: int, device=None) -> tuple[np.ndarray, list[torch.Tensor]]:
    """Grad-CAM of an ensemble for a batch of cropped (not yet enhanced) images.

    `members` is [(FundusNet, input size), ...]. Every model explains its own
    prediction at its own input size; the heatmaps are averaged at `out_size`
    and rescaled to [0, 1]. Returns (heatmaps [B, out_size, out_size], one
    logits tensor [B, outputs] per member, on the CPU).
    """
    total = np.zeros((len(cropped), out_size, out_size), np.float32)
    logits = []
    for model, size in members:
        x = torch.stack([to_tensor(model_input(c, size)) for c in cropped])
        cams, out = grad_cam(model, x if device is None else x.to(device))
        total += np.stack([resize_cam(c, out_size) for c in cams])
        logits.append(out.float().cpu())
    peak = total.reshape(len(cropped), -1).max(axis=1).reshape(-1, 1, 1)
    return np.where(peak > 0, total / np.maximum(peak, 1e-12), 0.0).astype(np.float32), logits


def lesion_overlap(cam: np.ndarray, lesion: np.ndarray, valid: np.ndarray, tolerance_frac: float = 0.04) -> dict:
    """Score how well a heatmap agrees with a lesion mask (all arrays HxW).

    pointing_hit   - heatmap maximum falls on a lesion (within a small tolerance)
    energy_in_mask - share of heatmap mass that lies on lesion pixels
    lesion_area    - share of the retina covered by lesions (the chance level for
                     energy_in_mask: a uniform heatmap scores exactly this)
    pixel_auc      - AUC of the heatmap as a per-pixel lesion detector (0.5 = chance)
    empty          - the heatmap is all zeros; it then scores as a miss / chance level

    Scores are NaN only when the mask has no lesion (or nothing but lesion) to compare with.
    """
    from .metrics import roc_auc

    lesion = lesion.astype(bool) & valid
    cam = np.where(valid, cam, 0.0).astype(np.float64)
    n_valid, n_lesion = int(valid.sum()), int(lesion.sum())
    area = n_lesion / max(n_valid, 1)
    if n_lesion == 0 or n_lesion == n_valid:
        return {"pointing_hit": np.nan, "energy_in_mask": np.nan, "lesion_area": area, "pixel_auc": np.nan, "empty": bool(cam.sum() <= 0)}
    if cam.sum() <= 0:
        return {"pointing_hit": 0.0, "energy_in_mask": area, "lesion_area": area, "pixel_auc": 0.5, "empty": True}
    k = max(1, int(round(tolerance_frac * cam.shape[0])))
    near = cv2.dilate(lesion.astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * k + 1, 2 * k + 1))) > 0
    peak = np.unravel_index(int(np.argmax(cam)), cam.shape)
    return {
        "pointing_hit": float(near[peak]),
        "energy_in_mask": float(cam[lesion].sum() / cam.sum()),
        "lesion_area": area,
        "pixel_auc": roc_auc(lesion[valid].astype(int), cam[valid]),
        "empty": False,
    }

In [ ]:
%%writefile fundalux/data.py
"""PyTorch datasets over the cache of cropped fundus images.

They live in the package (not the notebook) so DataLoader worker processes can
import them on any platform.
"""

from __future__ import annotations

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

from .degrade import MILD, SEVERE, degrade, random_degradation
from .models import to_tensor
from .preprocess import enhance_fundus, read_image


def worker_init(worker_id: int) -> None:
    """Keep OpenCV/torch from oversubscribing the CPU inside DataLoader workers."""
    cv2.setNumThreads(0)
    torch.set_num_threads(1)


def random_geometry(img: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Flip, rotate by any angle, small zoom and shift (the retina has no fixed 'up')."""
    size = img.shape[0]
    if rng.random() < 0.5:
        img = img[:, ::-1]
    c = (size - 1) / 2
    M = cv2.getRotationMatrix2D((c, c), float(rng.uniform(0, 360)), float(rng.uniform(0.9, 1.15)))
    M[:, 2] += rng.uniform(-0.03, 0.03, 2) * size
    return cv2.warpAffine(np.ascontiguousarray(img), M, (size, size), flags=cv2.INTER_LINEAR, borderValue=0)


def random_colour(img: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Per-channel gain and gamma jitter, imitating different cameras."""
    gamma = rng.uniform(0.8, 1.25)
    ramp = (np.arange(256, dtype=np.float32) / 255.0) ** gamma * 255.0
    out = np.empty_like(img)
    for ch, gain in enumerate(rng.uniform(0.9, 1.1, 3)):
        out[..., ch] = cv2.LUT(img[..., ch], np.clip(ramp * gain, 0, 255).astype(np.uint8))
    return out


class _CachedFundus(Dataset):
    """Shared loading and random-number handling.

    `fixed_degradation=(kind, severity)` applies the same degradation to every
    image; it is used to measure robustness, never for training.
    """

    def __init__(self, paths, size: int, train: bool, seed: int, fixed_degradation=None):
        self.paths = [str(p) for p in paths]
        self.size = size
        self.train = train
        self.seed = seed
        self.fixed_degradation = fixed_degradation
        self._train_rng = None

    def __len__(self) -> int:
        return len(self.paths)

    def _rng(self, index: int) -> np.random.Generator:
        if not self.train:  # evaluation is reproducible per image
            return np.random.default_rng((self.seed, index))
        if self._train_rng is None:  # one stream per worker process, reseeded each epoch by torch
            self._train_rng = np.random.default_rng(torch.initial_seed() % (2**32))
        return self._train_rng

    def _load(self, index: int) -> np.ndarray:
        img = read_image(self.paths[index])
        if img.shape[0] != self.size or img.shape[1] != self.size:
            img = cv2.resize(img, (self.size, self.size), interpolation=cv2.INTER_AREA)
        return img


class DRDataset(_CachedFundus):
    """Yields (enhanced image tensor, DR grade)."""

    def __init__(self, paths, grades, size: int, train: bool = False, degrade_p: float = 0.4, seed: int = 0, fixed_degradation=None):
        super().__init__(paths, size, train, seed, fixed_degradation)
        self.grades = np.asarray(grades, dtype=np.int64)
        self.degrade_p = degrade_p

    def __getitem__(self, index: int):
        rng = self._rng(index)
        img = self._load(index)
        if self.train:
            img = random_colour(random_geometry(img, rng), rng)
            if rng.random() < self.degrade_p:
                img, _, _ = random_degradation(img, rng, MILD)
        if self.fixed_degradation:
            img = degrade(img, *self.fixed_degradation, rng)
        return to_tensor(enhance_fundus(img)), int(self.grades[index])


class QualityDataset(_CachedFundus):
    """Yields (image tensor, 1.0 if ungradable else 0.0) for the quality gate.

    Images are *not* enhanced: CLAHE would hide the very defects the gate must see.
    Gradable images flagged `clean` are turned into synthetic ungradable ones with
    probability `severe_p`; any gradable image gets a mild, still-gradable
    degradation with probability `mild_p` so the gate does not over-reject.
    """

    def __init__(self, paths, labels, clean, size: int, input_size: int, train: bool = False, severe_p: float = 0.25, mild_p: float = 0.25, seed: int = 0, fixed_degradation=None,
                 gradable_range=MILD, ungradable_range=SEVERE):
        super().__init__(paths, size, train, seed, fixed_degradation)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.clean = np.asarray(clean, dtype=bool)
        self.input_size = input_size
        self.severe_p = severe_p
        self.mild_p = mild_p
        self.gradable_range, self.ungradable_range = gradable_range, ungradable_range

    def __getitem__(self, index: int):
        rng = self._rng(index)
        img = self._load(index)
        label = float(self.labels[index])
        if self.train:
            img = random_colour(random_geometry(img, rng), rng)
        if label == 0 and not self.fixed_degradation:
            u = rng.random()
            if self.clean[index] and u < self.severe_p:
                img, _, _ = random_degradation(img, rng, self.ungradable_range)
                label = 1.0
            elif u < self.severe_p + self.mild_p:
                img, _, _ = random_degradation(img, rng, self.gradable_range)
        if self.fixed_degradation:
            img = degrade(img, *self.fixed_degradation, rng)
        img = cv2.resize(img, (self.input_size, self.input_size), interpolation=cv2.INTER_AREA)
        return to_tensor(img), label

In [ ]:
%%writefile fundalux/datasets.py
"""Find images and labels inside the downloaded datasets and build one index.

The loaders search by file name instead of assuming fixed paths, because the
folder nesting differs between a Kaggle notebook Input, a download and a manual unzip.
Every loader returns a DataFrame with the columns in `COLUMNS`.
"""

from __future__ import annotations

import multiprocessing as mp
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from . import REFERABLE_MIN_GRADE
from .preprocess import _disc_u8, apply_crop, crop_fundus, find_fundus, keep_where, read_image
from .quality import quality_stats

IMAGE_EXT = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")
COLUMNS = ["source", "image_id", "path", "grade", "patient"]
TRAIN_SOURCES = ("eyepacs", "aptos")  # everything else is held out as external test data

# Bit flags inside the cached IDRiD lesion-mask PNGs.
LESION_BITS = {"MA": 1, "HE": 2, "EX": 4, "SE": 8, "OD": 16}
LESION_ANY = 1 | 2 | 4 | 8  # the optic disc is anatomy, not a lesion


def find_kaggle_input(slug: str, base="/kaggle/input") -> str | None:
    """Folder of a dataset attached to a Kaggle notebook ("Add Input"), or None.

    Kaggle has mounted datasets at /kaggle/input/<name> and, more recently, in
    deeper owner/name layouts, so the dataset name is searched a few levels down.
    """
    name, base = slug.split("/")[-1].lower(), Path(base)
    if not base.is_dir():
        return None
    for pattern in ("*", "*/*", "*/*/*"):
        for path in sorted(base.glob(pattern)):
            if path.is_dir() and path.name.lower() == name:
                return str(path)
    return None


def _files(root, suffixes) -> list[Path]:
    out = []
    for folder, _, names in os.walk(root):
        out.extend(Path(folder) / n for n in names if n.lower().endswith(suffixes))
    return sorted(out)


def _by_stem(paths) -> dict[str, Path]:
    return {p.stem.lower(): p for p in paths}


def _frame(source, ids, paths, grades, patients) -> pd.DataFrame:
    df = pd.DataFrame({"source": source, "image_id": list(ids), "path": [str(p) for p in paths], "grade": list(grades), "patient": list(patients)})
    return df.astype({"grade": int})[COLUMNS]


def index_eyepacs(root) -> pd.DataFrame:
    """EyePACS (Kaggle DR 2015 train set): trainLabels.csv with columns image, level."""
    csvs = [p for p in _files(root, (".csv",)) if p.name.lower() == "trainlabels.csv"]
    if not csvs:
        raise FileNotFoundError(f"trainLabels.csv not found under {root}")
    labels = pd.read_csv(csvs[0]).dropna(subset=["image", "level"])
    # The resized mirror ships a second, pre-cropped copy; we do our own cropping.
    images = _by_stem(p for p in _files(root, IMAGE_EXT) if "cropped" not in str(p.parent).lower())
    labels["path"] = labels["image"].astype(str).str.lower().map(images)
    labels = labels.dropna(subset=["path"])
    ids = labels["image"].astype(str)
    return _frame("eyepacs", ids, labels["path"], labels["level"], "eyepacs_" + ids.str.split("_").str[0])


def index_aptos(root) -> pd.DataFrame:
    """APTOS 2019: one or more CSVs with columns id_code, diagnosis."""
    frames = []
    for p in _files(root, (".csv",)):
        df = pd.read_csv(p)
        if {"id_code", "diagnosis"} <= set(df.columns):
            frames.append(df[["id_code", "diagnosis"]].dropna())
    if not frames:
        raise FileNotFoundError(f"No CSV with id_code/diagnosis columns under {root}")
    labels = pd.concat(frames).drop_duplicates("id_code")
    images = _by_stem(_files(root, IMAGE_EXT))
    labels["path"] = labels["id_code"].astype(str).str.lower().map(images)
    labels = labels.dropna(subset=["path"])
    ids = labels["id_code"].astype(str)
    return _frame("aptos", ids, labels["path"], labels["diagnosis"], "aptos_" + ids)


def index_idrid(root) -> pd.DataFrame:
    """IDRiD 'B. Disease Grading': training and testing sets reuse file names, so ids carry the set."""
    frames = []
    for csv in _files(root, (".csv",)):
        if "disease grading" not in csv.name.lower():
            continue
        subset = "train" if "training" in csv.name.lower() else "test"
        folder = "training set" if subset == "train" else "testing set"
        df = pd.read_csv(csv)
        df.columns = [str(c).strip().lower() for c in df.columns]
        df = df[["image name", "retinopathy grade"]].dropna()
        images = _by_stem(
            p for p in _files(root, IMAGE_EXT) if "disease grading" in str(p.parent).lower() and "original images" in str(p.parent).lower() and p.parent.name.lower().endswith(folder)
        )
        df["path"] = df["image name"].astype(str).str.strip().str.lower().map(images)
        df = df.dropna(subset=["path"])
        ids = subset + "_" + df["image name"].astype(str).str.strip()
        frames.append(_frame("idrid", ids, df["path"], df["retinopathy grade"], "idrid_" + ids))
    if not frames:
        raise FileNotFoundError(f"IDRiD disease-grading label CSVs not found under {root}")
    return pd.concat(frames, ignore_index=True)


def index_idrid_segmentation(root) -> list[dict]:
    """IDRiD 'A. Segmentation': [{image_id, path, masks: {MA|HE|EX|SE|OD: path}}]."""
    items = {}
    for p in _files(root, IMAGE_EXT):
        parent = str(p.parent).lower()
        if "segmentation" not in parent:
            continue
        if "original images" in parent:
            items.setdefault(p.stem, {"image_id": p.stem, "masks": {}})["path"] = str(p)
        elif "groundtruths" in parent:
            stem, _, kind = p.stem.rpartition("_")
            if kind in LESION_BITS:
                items.setdefault(stem, {"image_id": stem, "masks": {}})["masks"][kind] = str(p)
    return [items[k] for k in sorted(items) if "path" in items[k]]


def index_messidor(root) -> pd.DataFrame:
    """Messidor-2 with the adjudicated grades CSV. Images the adjudicators found
    ungradable have no grade; they get grade -1 and are kept for the quality gate."""
    csvs = [p for p in _files(root, (".csv",)) if "messidor" in p.name.lower()]
    if not csvs:
        raise FileNotFoundError(f"messidor_data.csv not found under {root}")
    df = pd.read_csv(csvs[0]).rename(columns={"image_id": "id_code", "adjudicated_dr_grade": "diagnosis"})
    images = _by_stem(_files(root, IMAGE_EXT))
    df["path"] = df["id_code"].astype(str).map(lambda s: images.get(Path(s).stem.lower()))
    df = df.dropna(subset=["path"])
    ids = df["id_code"].astype(str).map(lambda s: Path(s).stem)
    return _frame("messidor", ids, df["path"], df["diagnosis"].fillna(-1), "messidor_" + ids)


INDEXERS = {"eyepacs": index_eyepacs, "aptos": index_aptos, "idrid": index_idrid, "messidor": index_messidor}


def add_eyeq_labels(df: pd.DataFrame, eyeq_csvs) -> pd.DataFrame:
    """Attach EyeQ quality labels (0 good, 1 usable, 2 reject) to EyePACS rows; NaN = unlabelled."""
    df = df.copy()
    df["eyeq"] = np.nan
    frames = [pd.read_csv(p)[["image", "quality"]] for p in eyeq_csvs if p and os.path.exists(p)]
    if frames:
        eyeq = pd.concat(frames).drop_duplicates("image")
        lookup = dict(zip(eyeq["image"].astype(str).map(lambda s: Path(s).stem), eyeq["quality"]))
        is_eyepacs = df["source"] == "eyepacs"
        df.loc[is_eyepacs, "eyeq"] = df.loc[is_eyepacs, "image_id"].map(lookup)
    return df


def assign_splits(df: pd.DataFrame, seed: int = 42, val_frac: float = 0.1, test_frac: float = 0.1) -> pd.DataFrame:
    """Add `split` (train/val/test/external) and `referable` columns.

    Both eyes of a patient always land in the same split, and each split keeps
    the grade mix of every source (patients are stratified by their worst eye).
    """
    df = df.copy()
    df["referable"] = (df["grade"] >= REFERABLE_MIN_GRADE).astype(int)
    df["split"] = "external"
    pool = df[df["source"].isin(TRAIN_SOURCES)]
    patients = pool.groupby("patient").agg(source=("source", "first"), worst=("grade", "max")).sort_index()
    rng = np.random.default_rng(seed)
    assigned = {}
    for _, group in patients.groupby(["source", "worst"], sort=True):
        names = group.index.to_numpy().copy()
        rng.shuffle(names)
        n_test, n_val = int(round(len(names) * test_frac)), int(round(len(names) * val_frac))
        for i, name in enumerate(names):
            assigned[name] = "test" if i < n_test else "val" if i < n_test + n_val else "train"
    df.loc[pool.index, "split"] = pool["patient"].map(assigned)
    return df


def _cache_one(task):
    src, dst, size = task
    try:
        cropped, geom = crop_fundus(read_image(src), size)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if not cv2.imwrite(dst, cv2.cvtColor(cropped, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95]):
            raise OSError(f"could not write {dst}")
        return {"ok": True, "fundus_found": geom.found, **quality_stats(cropped)}
    except Exception as exc:  # a corrupt file must not kill a 40k-image job
        return {"ok": False, "fundus_found": False, "error": f"{type(exc).__name__}: {exc}"}


def build_cache(df: pd.DataFrame, cache_dir, size: int, workers: int | None = None, progress=None) -> pd.DataFrame:
    """Crop every image to the fundus at `size` and store it as JPEG under cache_dir/images.

    Returns the index with `cache_path` (relative to cache_dir), `fundus_found`
    and the quality statistics added; unreadable images are dropped.
    """
    df = df.reset_index(drop=True).copy()
    df["cache_path"] = [f"images/{s}/{i}.jpg" for s, i in zip(df["source"], df["image_id"])]
    tasks = [(src, os.path.join(cache_dir, rel), size) for src, rel in zip(df["path"], df["cache_path"])]
    workers = workers or os.cpu_count() or 1
    if workers > 1 and len(tasks) > 1:
        # "spawn" avoids the OpenCV-after-fork deadlock; _cache_one is importable, so it works in notebooks.
        with mp.get_context("spawn").Pool(workers, initializer=cv2.setNumThreads, initargs=(1,)) as pool:
            it = pool.imap(_cache_one, tasks, chunksize=16)
            results = list(progress(it, total=len(tasks)) if progress else it)
    else:
        it = map(_cache_one, tasks)
        results = list(progress(it, total=len(tasks)) if progress else it)
    df = pd.concat([df, pd.DataFrame(results)], axis=1)
    failed = df[~df["ok"]]
    if len(failed):
        print(f"Skipped {len(failed)} unreadable images, e.g. {failed['path'].iloc[0]}: {failed['error'].iloc[0]}")
    return df[df["ok"]].drop(columns=["ok", "error"], errors="ignore").reset_index(drop=True)


def _read_mask(path) -> np.ndarray:
    """Read an IDRiD ground-truth mask as a boolean array (any non-zero pixel is lesion)."""
    arr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if arr is None:
        from PIL import Image

        arr = np.asarray(Image.open(path).convert("L"))
    if arr.ndim == 3:
        arr = arr[..., :3].max(axis=2)
    return arr > 0


def cache_idrid_masks(items: list[dict], cache_dir, size: int, progress=None) -> pd.DataFrame:
    """Cache IDRiD segmentation images and their lesion masks with identical geometry.

    Each mask PNG holds one bit per lesion type (see LESION_BITS). A cached pixel
    is marked when lesions cover at least 5% of it, so microaneurysms survive the
    downscale.
    """
    rows = []
    for item in progress(items) if progress else items:
        img = read_image(item["path"])
        geom = find_fundus(img)
        cropped = keep_where(apply_crop(img, geom, size), _disc_u8(size))
        packed = np.zeros((size, size), np.uint8)
        for kind, path in item["masks"].items():
            mask = _read_mask(path)
            if mask.shape != img.shape[:2]:
                mask = cv2.resize(mask.astype(np.uint8), (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
            coverage = apply_crop(mask.astype(np.float32), geom, size, cv2.INTER_AREA if geom.side > size else cv2.INTER_LINEAR)
            packed[coverage >= 0.05] |= LESION_BITS[kind]
        rel_img, rel_mask = f"images/idrid_seg/{item['image_id']}.jpg", f"masks/idrid_seg/{item['image_id']}.png"
        for rel in (rel_img, rel_mask):
            os.makedirs(os.path.dirname(os.path.join(cache_dir, rel)), exist_ok=True)
        cv2.imwrite(os.path.join(cache_dir, rel_img), cv2.cvtColor(cropped, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])
        cv2.imwrite(os.path.join(cache_dir, rel_mask), packed)
        rows.append({"image_id": item["image_id"], "cache_path": rel_img, "mask_path": rel_mask, "lesion_types": "+".join(sorted(item["masks"]))})
    return pd.DataFrame(rows, columns=["image_id", "cache_path", "mask_path", "lesion_types"])

In [ ]:
%%writefile fundalux/synthetic.py
"""Synthetic fundus-like images and fake dataset folders.

Only for testing the code path end to end without the real data (unit tests and
the notebook's smoke-test mode). Nothing learned from these images is meaningful.
"""

from __future__ import annotations

import os

import cv2
import numpy as np

from .degrade import degrade


def synthetic_fundus(rng: np.random.Generator, width: int = 240, height: int = 160, grade: int = 0, ungradable: bool = False):
    """Draw a retina-like disc; higher grades get more dark and bright spots.

    Returns (image RGB uint8, lesions) where lesions maps "HE"/"EX" to boolean masks.
    """
    yy, xx = np.mgrid[:height, :width].astype(np.float32)
    cx, cy, r = width / 2, height / 2, 0.46 * width  # wider than tall: the disc is cut off top and bottom
    dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
    inside = dist < r
    shade = 1.0 - 0.35 * (dist / r) ** 2
    base = np.array([190.0, 95.0, 45.0]) * rng.uniform(0.8, 1.1)
    img = np.zeros((height, width, 3), np.float32)
    img[inside] = (shade[..., None] * base)[inside]
    # Optic disc and a few vessels radiating from it.
    side = 1 if rng.random() < 0.5 else -1
    od = (int(cx + side * 0.5 * r), int(cy + rng.uniform(-0.05, 0.05) * r))
    cv2.circle(img, od, max(2, int(0.13 * r)), (245.0, 215.0, 150.0), -1)
    for _ in range(6):
        pts, angle = [od], rng.uniform(0, 2 * np.pi)
        for _ in range(6):
            angle += rng.uniform(-0.5, 0.5)
            pts.append((int(pts[-1][0] + 0.25 * r * np.cos(angle)), int(pts[-1][1] + 0.25 * r * np.sin(angle))))
        cv2.polylines(img, [np.array(pts, np.int32)], False, (120.0, 40.0, 25.0), max(1, int(0.015 * r)))
    lesions = {"HE": np.zeros((height, width), np.uint8), "EX": np.zeros((height, width), np.uint8)}
    for _ in range(3 * grade):
        rho, phi = 0.8 * r * np.sqrt(rng.uniform()), rng.uniform(0, 2 * np.pi)
        centre = (int(cx + rho * np.cos(phi)), int(cy + rho * np.sin(phi)))
        radius = max(1, int(rng.uniform(0.02, 0.05) * r))
        kind, colour = ("HE", (95.0, 15.0, 15.0)) if rng.random() < 0.6 else ("EX", (250.0, 225.0, 120.0))
        cv2.circle(img, centre, radius, colour, -1)
        cv2.circle(lesions[kind], centre, radius, 1, -1)
    img[~inside] = 0
    img = np.clip(cv2.GaussianBlur(img, (0, 0), 0.6) + rng.normal(0, 2.0, img.shape), 0, 255).astype(np.uint8)
    img[~inside] = 0
    if ungradable:
        img = degrade(img, "blur", 1.0, rng)
    return img, {k: (v > 0) & inside for k, v in lesions.items()}


def _save(path, img_rgb) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    cv2.imwrite(path, cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))


def _grades(rng, n):
    return rng.choice(5, size=n, p=[0.45, 0.1, 0.25, 0.1, 0.1])


def make_fake_datasets(root, n_eyepacs_patients: int = 40, n_aptos: int = 40, n_idrid: int = 16, n_idrid_seg: int = 6, n_messidor: int = 16, seed: int = 0) -> dict:
    """Write tiny datasets that mirror the folder layout of the real Kaggle downloads.

    Returns {name: folder} plus "eyeq_csv": path of a fake EyeQ label file.
    """
    rng = np.random.default_rng(seed)
    roots = {k: os.path.join(root, k) for k in ("eyepacs", "aptos", "idrid", "messidor")}

    # EyePACS (tanlikesmath/diabetic-retinopathy-resized) + EyeQ quality labels
    rows, eyeq = [], []
    for pid in range(10, 10 + n_eyepacs_patients):
        for eye in ("left", "right"):
            grade = int(_grades(rng, 1)[0])
            bad = rng.random() < 0.2
            img, _ = synthetic_fundus(rng, grade=grade, ungradable=bad)
            name = f"{pid}_{eye}"
            _save(os.path.join(roots["eyepacs"], "resized_train", "resized_train", name + ".jpeg"), img)
            rows.append(f"{name},{grade}")
            if rng.random() < 0.8:
                eyeq.append(f"{len(eyeq)},{name}.jpeg,{2 if bad else int(rng.integers(2))},{grade}")
    _save(os.path.join(roots["eyepacs"], "resized_train_cropped", "resized_train_cropped", "10_left.jpeg"), img)
    with open(os.path.join(roots["eyepacs"], "trainLabels.csv"), "w") as f:
        f.write("image,level\n" + "\n".join(rows) + "\n")
    with open(os.path.join(roots["eyepacs"], "trainLabels_cropped.csv"), "w") as f:
        f.write(",Unnamed: 0,image,level\n0,0,10_left,0\n")
    eyeq_csv = os.path.join(root, "Label_EyeQ_train.csv")
    with open(eyeq_csv, "w") as f:
        f.write(",image,quality,DR_grade\n" + "\n".join(eyeq) + "\n")

    # APTOS 2019 (mariaherrerot/aptos2019): three folders, three CSVs, trailing blank lines
    parts = {"train_1.csv": ("train_images", []), "valid.csv": ("val_images", []), "test.csv": ("test_images", [])}
    for i, grade in enumerate(_grades(rng, n_aptos)):
        csv = "train_1.csv" if i % 5 < 3 else "valid.csv" if i % 5 == 3 else "test.csv"
        folder, lines = parts[csv]
        name = f"{int(rng.integers(16**12)):012x}"
        img, _ = synthetic_fundus(rng, width=200, height=200, grade=int(grade))
        _save(os.path.join(roots["aptos"], folder, folder, name + ".png"), img)
        lines.append(f"{name},{grade}")
    for csv, (_, lines) in parts.items():
        with open(os.path.join(roots["aptos"], csv), "w") as f:
            f.write("id_code,diagnosis\r\n" + "\r\n".join(lines) + "\r\n\r\n\r\n")

    # IDRiD (aaryapatel98/indian-diabetic-retinopathy-image-dataset): grading + segmentation
    grading = os.path.join(roots["idrid"], "B.%20Disease%20Grading", "B. Disease Grading")
    for folder, csv, n in (("a. Training Set", "a. IDRiD_Disease Grading_Training Labels.csv", n_idrid - n_idrid // 4), ("b. Testing Set", "b. IDRiD_Disease Grading_Testing Labels.csv", n_idrid // 4)):
        lines = []
        for i, grade in enumerate(_grades(rng, n), start=1):
            img, _ = synthetic_fundus(rng, grade=int(grade))
            _save(os.path.join(grading, "1. Original Images", folder, f"IDRiD_{i:03d}.jpg"), img)
            lines.append(f"IDRiD_{i:03d},{grade},0,,,")
        os.makedirs(os.path.join(grading, "2. Groundtruths"), exist_ok=True)
        with open(os.path.join(grading, "2. Groundtruths", csv), "w") as f:
            f.write("Image name,Retinopathy grade,Risk of macular edema ,,,\r\n" + "\r\n".join(lines) + "\r\n")
    seg = os.path.join(roots["idrid"], "A.%20Segmentation", "A. Segmentation")
    mask_dirs = {"HE": "2. Haemorrhages", "EX": "3. Hard Exudates"}
    for i in range(1, n_idrid_seg + 1):
        folder = "a. Training Set" if i <= n_idrid_seg - 2 else "b. Testing Set"
        img, lesions = synthetic_fundus(rng, grade=int(rng.integers(2, 5)))
        _save(os.path.join(seg, "1. Original Images", folder, f"IDRiD_{i:02d}.jpg"), img)
        for kind, mask in lesions.items():
            if mask.any():
                path = os.path.join(seg, "2. All Segmentation Groundtruths", folder, mask_dirs[kind], f"IDRiD_{i:02d}_{kind}.tif")
                os.makedirs(os.path.dirname(path), exist_ok=True)
                cv2.imwrite(path, mask.astype(np.uint8) * 76)

    # Messidor-2 (mariaherrerot/messidor2preprocess): a few adjudicated-ungradable rows have no grade
    lines = []
    for i, grade in enumerate(_grades(rng, n_messidor)):
        gradable = i % 8 != 7
        img, _ = synthetic_fundus(rng, width=200, height=200, grade=int(grade), ungradable=not gradable)
        name = f"2005{i:04d}_{int(rng.integers(10**5)):05d}_0100_PP.png"
        _save(os.path.join(roots["messidor"], "messidor-2", "messidor-2", "preprocess", name), img)
        lines.append(f"{name},{grade},0,1" if gradable else f"{name},,,0")
    with open(os.path.join(roots["messidor"], "messidor_data.csv"), "w") as f:
        f.write("id_code,diagnosis,adjudicated_dme,adjudicated_gradable\n" + "\n".join(lines) + "\n")

    return {**roots, "eyeq_csv": eyeq_csv}

In [ ]:
%%writefile fundalux/pipeline.py
"""End-to-end inference: quality gate -> DR classifier -> Grad-CAM -> advice."""

from __future__ import annotations

import json
from dataclasses import dataclass, field, replace
from pathlib import Path

import cv2
import numpy as np

from . import GRADE_NAMES
from .heatmap import palette_overlay, resize_cam
from .inference import ensemble_probabilities, load_backend, to_input
from .preprocess import crop_fundus, model_input
from .quality import basic_checks, quality_hints, quality_stats

DISCLAIMER = "Screening aid only - not a diagnosis. A qualified clinician must confirm every result."


@dataclass
class Prediction:
    gradable: bool
    ungradable_score: float | None  # quality gate output, P(ungradable)
    reasons: list[str] = field(default_factory=list)  # why the image was rejected
    quality: dict = field(default_factory=dict)  # raw image statistics
    referable: bool | None = None
    referable_score: float | None = None  # P(referable DR)
    threshold: float | None = None
    site: str | None = None  # whose calibrated threshold was used; None = the bundle's global threshold
    grade: int | None = None
    grade_probs: list[float] | None = None
    member_scores: dict[str, float] | None = None  # P(referable DR) of each model in the ensemble
    advice: str = ""
    cropped: np.ndarray | None = None  # what the quality gate saw
    enhanced: np.ndarray | None = None  # what the DR classifier saw
    heatmap: np.ndarray | None = None  # Grad-CAM in [0, 1], same size as `enhanced`
    overlay: np.ndarray | None = None

    @property
    def grade_name(self) -> str | None:
        return None if self.grade is None else GRADE_NAMES[self.grade]

    def with_threshold(self, threshold: float, site: str | None = None) -> "Prediction":
        """The same analysis decided at another threshold, e.g. a site's - no re-scoring needed."""
        out = replace(self, threshold=float(threshold), site=site)
        if self.referable_score is not None:
            out.referable = self.referable_score >= out.threshold
            if self.gradable:
                out.advice = referral_advice(True, out.referable, self.grade)
        return out

    def summary(self) -> dict:
        """JSON-friendly result without the image arrays."""
        keys = ("gradable", "ungradable_score", "reasons", "referable", "referable_score", "threshold", "site", "grade", "grade_probs", "member_scores", "advice")
        out = {k: getattr(self, k) for k in keys}
        out["grade_name"] = self.grade_name
        return out


def referral_advice(gradable: bool, referable: bool | None, grade: int | None) -> str:
    if not gradable:
        return "Image rejected - retake the photograph. If it is still ungradable after two more attempts, refer the patient for an in-person eye examination."
    if referable:
        if grade is not None and grade >= 3:
            return "Referable DR suspected, possibly severe - refer to an ophthalmologist urgently."
        return "Referable DR suspected - refer to an ophthalmologist."
    return "No referable DR detected - advise routine re-screening in 12 months, sooner if vision changes."


class Fundalux:
    """Loads the trained models and screens one photograph at a time.

    The DR classifier is an ensemble: every member scores the image and their
    probabilities are averaged. A bundle with a single model is an ensemble of one.
    Models run in ONNX Runtime when the bundle has an ONNX export (python -m
    fundalux.export), otherwise in PyTorch.
    """

    def __init__(self, members, gate, config: dict, backend: str = ""):
        self.members, self.gate, self.config, self.backend = members, gate, config, backend
        self.dr_members = [(m.name, m, m.size) for m in members]  # (architecture, member, input size)
        self.image_size = int(config["image_size"])
        self.dr_threshold = float(config["dr"]["threshold"])
        self.quality_size = int(config["quality"]["input_size"])
        self.quality_threshold = float(config["quality"]["threshold"])
        self.hint_ranges = config["quality"].get("hint_ranges")
        self.min_input_side = int(config.get("min_input_side", 224))

    @classmethod
    def load(cls, model_dir, backend: str = "auto") -> "Fundalux":
        model_dir = Path(model_dir)
        config_path = model_dir / "config.json"
        if not config_path.exists():
            raise FileNotFoundError(
                f"{config_path} not found. Train the models with the Kaggle notebook and unzip fundalux_outputs.zip into {model_dir}/ (see models/README.md)."
            )
        config = json.loads(config_path.read_text())
        name, members, gate = load_backend(model_dir, config, backend)
        return cls(members, gate, config, name)

    def ungradable_score(self, cropped: np.ndarray) -> float:
        small = cv2.resize(cropped, (self.quality_size, self.quality_size), interpolation=cv2.INTER_AREA)
        return self.gate.ungradable(to_input([small]))

    def _gate(self, image: np.ndarray) -> Prediction:
        """Crop the photograph and run the quality gate (rule checks, then the learned model)."""
        cropped, geom = crop_fundus(image, self.image_size)
        stats = quality_stats(cropped)
        reasons = basic_checks(image, geom, stats, self.min_input_side)
        score = self.ungradable_score(cropped) if geom.found else None
        if score is not None and score >= self.quality_threshold:
            reasons.append(f"Image quality is too poor to grade (ungradable score {score:.2f}, limit {self.quality_threshold:.2f}).")
        gradable = not reasons
        if not gradable:
            reasons.extend(quality_hints(stats, self.hint_ranges))
        return Prediction(gradable=gradable, ungradable_score=score, reasons=reasons, quality=stats, cropped=cropped)

    def _score(self, cropped: np.ndarray, with_cam: bool):
        """Run every member at its own input size. Returns (logits per member, averaged heatmap or None)."""
        logits, total = [], np.zeros((self.image_size, self.image_size), np.float32)
        for m in self.members:
            out, cams = m.run(to_input([model_input(cropped, m.size)]), with_cam)
            logits.append(out)
            if with_cam:
                total += resize_cam(cams[0], self.image_size)
        heatmap = None
        if with_cam:
            peak = float(total.max())
            heatmap = total / peak if peak > 0 else total
        return logits, heatmap

    def referable_score(self, image: np.ndarray) -> tuple[bool, float | None]:
        """(passes the quality gate, P(referable DR) or None) without Grad-CAM - for calibrating a site."""
        gate = self._gate(image)
        if not gate.gradable:
            return False, None
        logits, _ = self._score(gate.cropped, with_cam=False)
        return True, float(ensemble_probabilities(logits)[0][0])

    def predict(self, image: np.ndarray, skip_gate: bool = False, site=None) -> Prediction:
        """Screen one RGB uint8 photograph.

        `site` is a calibrated SiteProfile (see sites.py); its threshold replaces the
        bundle's global one. With `skip_gate` the classifier runs even on a rejected
        image (the result keeps gradable=False) - only for demonstrating why the gate exists.
        """
        result = self._gate(image)
        gradable, cropped = result.gradable, result.cropped
        if not gradable and not skip_gate:
            result.advice = referral_advice(False, None, None)
            return result

        logits, heatmap = self._score(cropped, with_cam=True)
        referable, grades, per_member = ensemble_probabilities(logits)
        result.enhanced = model_input(cropped, self.image_size)
        result.referable_score = float(referable[0])
        result.member_scores = {m.name: float(p[0]) for m, p in zip(self.members, per_member)}
        result.threshold = self.dr_threshold if site is None else float(site.threshold)
        result.site = None if site is None else site.name
        result.referable = result.referable_score >= result.threshold
        if grades is not None:
            result.grade_probs = [float(p) for p in grades[0]]
            result.grade = int(np.argmax(grades[0]))
        result.heatmap = heatmap
        result.overlay = palette_overlay(result.enhanced, heatmap)
        result.advice = referral_advice(gradable, result.referable, result.grade)
        return result

In [ ]:
%%writefile fundalux/sites.py
"""Per-site decision thresholds.

The classifier ranks images well at every site, but its scores sit on a
different scale for each camera, so one threshold does not transfer. A site
profile stores a threshold calibrated on graded images from that site, tuned
for the same target sensitivity as the global threshold.

Profiles live in <model dir>/sites.json. A threshold only means something for
the model that produced the scores, so the file records a fingerprint of the
model bundle; profiles from another bundle are kept but marked stale.
"""

from __future__ import annotations

import hashlib
import json
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np

from . import REFERABLE_MIN_GRADE
from .metrics import crossfit_operating_point, sens_spec, threshold_for_sensitivity

SITES_FILE = "sites.json"
MIN_PER_CLASS = 20  # below this a site cannot be calibrated at all
RECOMMENDED_PER_CLASS = 50  # below this the profile carries a warning


class CalibrationError(ValueError):
    pass


@dataclass
class SiteProfile:
    name: str
    threshold: float
    target_sensitivity: float
    n_images: int  # gate-accepted images the threshold was fitted on
    n_referable: int
    expected_sensitivity: float  # on new images from this site, estimated without reusing the fitting images
    expected_specificity: float
    sensitivity_range: tuple[float, float] = (float("nan"), float("nan"))
    specificity_range: tuple[float, float] = (float("nan"), float("nan"))
    estimate: str = ""  # how the expected values were obtained
    source: str = ""  # "built-in" for profiles written with the model bundle
    rejected_by_gate: int = 0
    created: str = field(default_factory=lambda: time.strftime("%Y-%m-%d"))
    stale: bool = False  # fitted for a different model bundle

    @property
    def warnings(self) -> list[str]:
        out = []
        if min(self.n_referable, self.n_images - self.n_referable) < RECOMMENDED_PER_CLASS:
            out.append(f"Fewer than {RECOMMENDED_PER_CLASS} images in one class: the threshold is imprecise.")
        if self.stale:
            out.append("Calibrated for a different model - recalibrate before use.")
        return out


def model_fingerprint(config: dict) -> str:
    """Identifies the scoring models; a new training run gives a new fingerprint."""
    dr = config["dr"]
    members = dr.get("models") or [{"arch": dr["arch"], "weights": dr["weights"], "input_size": config["image_size"]}]
    key = json.dumps({"members": members, "trained": config.get("trained_with", {}), "threshold": dr["threshold"]}, sort_keys=True)
    return hashlib.sha256(key.encode()).hexdigest()[:16]


def patient_groups(source: str, image_ids) -> np.ndarray:
    """Group both eyes of one patient where the file names allow it (EyePACS: 123_left / 123_right)."""
    ids = np.asarray(image_ids, dtype=str)
    if source == "eyepacs":
        return np.array([i.rsplit("_", 1)[0] for i in ids])
    return ids


def calibrate(name: str, referable, scores, target_sensitivity: float, groups=None, source: str = "", rejected_by_gate: int = 0) -> SiteProfile:
    """Fit a site threshold on gate-accepted images; estimate its accuracy by cross-fitting."""
    y = np.asarray(referable).astype(bool)
    s = np.asarray(scores, dtype=np.float64)
    if y.size != s.size:
        raise CalibrationError("Labels and scores have different lengths.")
    n_pos, n_neg = int(y.sum()), int((~y).sum())
    if min(n_pos, n_neg) < MIN_PER_CLASS:
        raise CalibrationError(
            f"Calibration needs at least {MIN_PER_CLASS} referable and {MIN_PER_CLASS} non-referable gradable images; got {n_pos} and {n_neg}."
        )
    estimate = crossfit_operating_point(y, s, target_sensitivity, groups=groups)
    return SiteProfile(
        name=name,
        threshold=threshold_for_sensitivity(y, s, target_sensitivity),
        target_sensitivity=target_sensitivity,
        n_images=int(y.size),
        n_referable=n_pos,
        expected_sensitivity=estimate["sensitivity"],
        expected_specificity=estimate["specificity"],
        sensitivity_range=estimate["sensitivity_range"],
        specificity_range=estimate["specificity_range"],
        estimate=f"cross-fitted: tuned on one half of the site's images, tested on the other ({estimate['repeats']} random splits)",
        source=source,
        rejected_by_gate=rejected_by_gate,
    )


def calibrate_on_split(name: str, fit: dict, test: dict, target_sensitivity: float, source: str = "") -> SiteProfile:
    """Fit on one set of images and measure on a separate one (used when a site has a validation split).

    `fit` and `test` are {"referable": ..., "scores": ...}.
    """
    profile = calibrate(name, fit["referable"], fit["scores"], target_sensitivity, source=source)
    sens, spec = sens_spec(test["referable"], test["scores"], profile.threshold)
    profile.expected_sensitivity, profile.expected_specificity = sens, spec
    profile.sensitivity_range = profile.specificity_range = (float("nan"), float("nan"))
    profile.estimate = f"tuned on {profile.n_images} validation images, measured on {len(test['scores'])} separate test images"
    return profile


# Display names for the datasets the bundle is evaluated on.
BUILTIN_NAMES = {
    "eyepacs": "EyePACS (USA, many clinics)",
    "aptos": "APTOS 2019 (Aravind, India)",
    "idrid": "IDRiD (Nanded, India)",
    "messidor": "Messidor-2 (France)",
}


def builtin_profiles(predictions, target_sensitivity: float) -> list[SiteProfile]:
    """Profiles for the evaluation datasets, from the bundle's per-image predictions.

    `predictions` needs columns source, image_id, split, grade, gate_pass, p_ref.
    Sources with a validation split are tuned on it and measured on their test
    split; external sources are tuned on all their images and measured by cross-fitting.
    """
    df = predictions[(predictions["grade"] >= 0) & predictions["gate_pass"].astype(bool) & predictions["p_ref"].notna()]
    profiles = []
    for source, name in BUILTIN_NAMES.items():
        part = df[df["source"] == source]
        if part.empty:
            continue
        referable = (part["grade"] >= REFERABLE_MIN_GRADE).to_numpy()
        try:
            val, test = part[part["split"] == "val"], part[part["split"] == "test"]
            if len(val) and len(test):
                pack = lambda f: {"referable": (f["grade"] >= REFERABLE_MIN_GRADE).to_numpy(), "scores": f["p_ref"].to_numpy()}
                profiles.append(calibrate_on_split(name, pack(val), pack(test), target_sensitivity, source="built-in"))
            else:
                groups = patient_groups(source, part["image_id"])
                profiles.append(calibrate(name, referable, part["p_ref"].to_numpy(), target_sensitivity, groups=groups, source="built-in"))
        except CalibrationError:
            continue
    return profiles


class SiteRegistry:
    """The site profiles stored next to a model bundle."""

    def __init__(self, path: Path, fingerprint: str, profiles: list[SiteProfile]):
        self.path, self.fingerprint, self.profiles = Path(path), fingerprint, profiles

    @classmethod
    def load(cls, model_dir, config: dict) -> "SiteRegistry":
        path, fingerprint = Path(model_dir) / SITES_FILE, model_fingerprint(config)
        profiles = []
        if path.exists():
            data = json.loads(path.read_text())
            for raw in data.get("sites", []):
                raw = {k: tuple(v) if k.endswith("_range") and v is not None else v for k, v in raw.items() if k in SiteProfile.__dataclass_fields__}
                raw = {k: (float("nan"), float("nan")) if k.endswith("_range") and v is None else v for k, v in raw.items()}
                profile = SiteProfile(**raw)
                profile.stale = data.get("model_fingerprint") != fingerprint or raw.get("stale", False)
                profiles.append(profile)
        return cls(path, fingerprint, profiles)

    def get(self, name: str) -> SiteProfile | None:
        return next((p for p in self.profiles if p.name == name), None)

    def usable(self) -> list[SiteProfile]:
        return [p for p in self.profiles if not p.stale]

    def put(self, profile: SiteProfile) -> None:
        self.profiles = [p for p in self.profiles if p.name != profile.name] + [profile]
        self.save()

    def remove(self, name: str) -> None:
        self.profiles = [p for p in self.profiles if p.name != name]
        self.save()

    def save(self) -> None:
        def clean(value):
            if isinstance(value, float) and not np.isfinite(value):
                return None
            if isinstance(value, tuple):
                return [clean(v) for v in value]
            return value

        sites = [{k: clean(v) for k, v in asdict(p).items() if k != "stale"} for p in self.profiles if not p.stale]
        stale = [{**{k: clean(v) for k, v in asdict(p).items()}, "stale": True} for p in self.profiles if p.stale]
        tmp = self.path.with_suffix(".tmp")
        tmp.write_text(json.dumps({"model_fingerprint": self.fingerprint, "sites": sites + stale}, indent=2))
        tmp.replace(self.path)

In [ ]:
%%writefile fundalux/inference.py
"""Inference backends: ONNX Runtime (default, no PyTorch needed) and PyTorch.

Both expose the same small interface, so the pipeline does not care which one
runs. Each DR network is exported with two outputs, its last feature map and
its logits. Because the head after that feature map is global-average pooling
(optionally a LayerNorm, for ConvNeXt) and one linear layer, the Grad-CAM
gradient has a closed form, so heatmaps need no automatic differentiation:

    linear head:     d logit / d F[c,h,w] = W[c] / HW
    LayerNorm head:  d logit / d p = (g - mean(g) - p_hat * mean(g * p_hat)) / sigma,  g = W * gamma,
                     with p the pooled features, then divided by HW

Grad-CAM weights each channel by the spatial mean of that gradient, so this is
exactly what torch.autograd gives (tests/test_inference.py checks it).
"""

from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np

HEADS_FILE = "heads.npz"


def to_input(images: list[np.ndarray]) -> np.ndarray:
    """RGB uint8 HxWx3 images (same size) -> float32 NCHW batch in 0-255; the networks normalise internally."""
    return np.ascontiguousarray(np.stack(images).transpose(0, 3, 1, 2), dtype=np.float32)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.asarray(z, dtype=np.float64)))


def ensemble_probabilities(logits) -> tuple[np.ndarray, np.ndarray | None, np.ndarray]:
    """Combine the members of an ensemble by averaging their probabilities.

    `logits` is a list with one [N, 1 + grades] array (or tensor) per model.
    Returns (P(positive) [N], grade probabilities [N, grades] or None, each member's P(positive) [members, N]).
    """
    arrays = []
    for x in logits:
        if hasattr(x, "detach"):  # a torch tensor
            x = x.detach().cpu().numpy()
        arrays.append(np.asarray(x, dtype=np.float64))
    positive = np.stack([sigmoid(x[:, 0]) for x in arrays])
    grades = None
    if arrays[0].shape[1] > 1:
        shifted = [x[:, 1:] - x[:, 1:].max(axis=1, keepdims=True) for x in arrays]
        grades = np.mean([np.exp(s) / np.exp(s).sum(axis=1, keepdims=True) for s in shifted], axis=0)
    return positive.mean(axis=0), grades, positive


def normalise_cams(cams: np.ndarray) -> np.ndarray:
    """ReLU, then scale each map to a peak of 1 (an all-zero map stays zero)."""
    cams = np.maximum(cams, 0.0)
    peak = cams.reshape(len(cams), -1).max(axis=1).reshape(-1, 1, 1)
    return np.where(peak > 1e-12, cams / np.maximum(peak, 1e-12), 0.0).astype(np.float32)


def cam_from_features(feats: np.ndarray, head: dict, target: int = 0) -> np.ndarray:
    """Grad-CAM of logit `target` from a feature map [B, C, h, w] and the head parameters."""
    feats = feats.astype(np.float64)
    w = head["weight"][target].astype(np.float64)  # [C]
    if head["kind"] == "layernorm":
        pooled = feats.mean(axis=(2, 3))  # [B, C]
        mu = pooled.mean(axis=1, keepdims=True)
        sigma = np.sqrt(((pooled - mu) ** 2).mean(axis=1, keepdims=True) + float(head["eps"]))
        p_hat = (pooled - mu) / sigma
        g = w * head["gamma"].astype(np.float64)  # [C]
        weights = (g - g.mean() - p_hat * (g * p_hat).mean(axis=1, keepdims=True)) / sigma  # [B, C]
    else:
        weights = np.broadcast_to(w, (feats.shape[0], w.size))
    # The 1/HW factor is common to every channel and disappears in the normalisation.
    return normalise_cams(np.einsum("bc,bchw->bhw", weights, feats))


class OnnxMember:
    """One DR network in ONNX Runtime."""

    def __init__(self, name: str, path, size: int, head: dict, session_options=None):
        import onnxruntime as ort

        self.name, self.size, self.head = name, int(size), head
        self.session = ort.InferenceSession(str(path), sess_options=session_options, providers=["CPUExecutionProvider"])

    def run(self, x: np.ndarray, with_cam: bool) -> tuple[np.ndarray, np.ndarray | None]:
        """x: float32 NCHW. Returns (logits [B, outputs], heatmaps [B, h, w] or None)."""
        feats, logits = self.session.run(["features", "logits"], {"image": x})
        return logits, (cam_from_features(feats, self.head) if with_cam else None)


class OnnxGate:
    def __init__(self, path, session_options=None):
        import onnxruntime as ort

        self.session = ort.InferenceSession(str(path), sess_options=session_options, providers=["CPUExecutionProvider"])

    def ungradable(self, x: np.ndarray) -> float:
        return float(sigmoid(self.session.run(["logits"], {"image": x})[0][0, 0]))


class TorchMember:
    """One DR network in PyTorch (used for training-time checks and bundles without ONNX)."""

    def __init__(self, name: str, model, size: int):
        self.name, self.model, self.size = name, model.eval(), int(size)

    def run(self, x: np.ndarray, with_cam: bool):
        import torch

        from .gradcam import grad_cam

        xt = torch.from_numpy(x)
        if with_cam:
            cams, out = grad_cam(self.model, xt)
            return out.float().numpy(), cams
        with torch.no_grad():
            return self.model(xt).float().numpy(), None


class TorchGate:
    def __init__(self, model):
        self.model = model.eval()

    def ungradable(self, x: np.ndarray) -> float:
        import torch

        with torch.no_grad():
            return float(torch.sigmoid(self.model(torch.from_numpy(x))[0, 0]))


def session_options():
    import onnxruntime as ort

    opts = ort.SessionOptions()
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    # The networks run one after another, each in its own session. Spinning (busy-waiting) worker
    # threads would keep every idle session burning CPU and slow down the one that is working.
    opts.add_session_config_entry("session.intra_op.allow_spinning", "0")
    opts.add_session_config_entry("session.inter_op.allow_spinning", "0")
    threads = int(os.environ.get("FUNDALUX_THREADS", "0"))  # 0 = let ONNX Runtime decide
    if threads:
        opts.intra_op_num_threads = threads
    return opts


def onnx_available(model_dir, config: dict) -> bool:
    spec = config.get("onnx")
    if not spec:
        return False
    try:
        import onnxruntime  # noqa: F401
    except ImportError:
        return False
    folder = Path(model_dir) / spec["dir"]
    files = [m["file"] for m in spec["members"]] + [spec["quality"], spec["heads"]]
    return all((folder / f).exists() for f in files)


def load_onnx(model_dir, config: dict):
    """(members, gate) from the bundle's ONNX export (see fundalux/export.py)."""
    spec, folder = config["onnx"], Path(model_dir) / config["onnx"]["dir"]
    heads = np.load(folder / spec["heads"], allow_pickle=False)
    opts = session_options()
    members = []
    for i, m in enumerate(spec["members"]):
        head = {"weight": heads[f"m{i}_weight"], "kind": str(heads[f"m{i}_kind"]), "gamma": heads[f"m{i}_gamma"], "eps": float(heads[f"m{i}_eps"])}
        members.append(OnnxMember(m["arch"], folder / m["file"], m["input_size"], head, opts))
    return members, OnnxGate(folder / spec["quality"], opts)


def load_torch(model_dir, config: dict):
    from .models import FundusNet, load_weights

    model_dir, dr = Path(model_dir), config["dr"]
    # Bundles from the first, single-model notebook describe their one model directly under "dr".
    specs = dr.get("models") or [{"arch": dr["arch"], "weights": dr["weights"], "input_size": config["image_size"]}]
    members = [TorchMember(s["arch"], load_weights(FundusNet(s["arch"], num_grades=dr["num_grades"]), model_dir / s["weights"]), s["input_size"]) for s in specs]
    gate = TorchGate(load_weights(FundusNet(config["quality"]["arch"], num_grades=0), model_dir / config["quality"]["weights"]))
    return members, gate


def load_backend(model_dir, config: dict, backend: str = "auto"):
    """Returns (backend name, members, gate). `backend` is auto, onnx or torch (env FUNDALUX_BACKEND overrides auto)."""
    if backend == "auto":
        backend = os.environ.get("FUNDALUX_BACKEND", "auto")
    if backend == "onnx" or (backend == "auto" and onnx_available(model_dir, config)):
        return ("onnx", *load_onnx(model_dir, config))
    return ("torch", *load_torch(model_dir, config))


def read_config(model_dir) -> dict:
    return json.loads((Path(model_dir) / "config.json").read_text())

In [ ]:
%%writefile fundalux/heatmap.py
"""Heatmap drawing (no PyTorch): resizing and overlays for Grad-CAM maps in [0, 1]."""

from __future__ import annotations

import cv2
import numpy as np


def resize_cam(cam: np.ndarray, size: int) -> np.ndarray:
    return np.clip(cv2.resize(cam.astype(np.float32), (size, size), interpolation=cv2.INTER_CUBIC), 0.0, 1.0)


def overlay_heatmap(img: np.ndarray, cam: np.ndarray, alpha: float = 0.5) -> np.ndarray:
    """Classic JET overlay (used in the training figures); weak activations stay transparent."""
    cam = resize_cam(cam, img.shape[0]) if cam.shape[:2] != img.shape[:2] else cam
    colour = cv2.cvtColor(cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    w = (alpha * cam)[..., None]
    return np.clip(img * (1 - w) + colour * w, 0, 255).astype(np.uint8)


def _lut(stops: list[tuple[float, str]]) -> np.ndarray:
    xs = np.linspace(0, 1, 256)
    rgb = [tuple(int(h[i : i + 2], 16) for i in (1, 3, 5)) for _, h in stops]
    pos = [p for p, _ in stops]
    return np.stack([np.interp(xs, pos, [c[k] for c in rgb]) for k in range(3)], axis=1)


# Blue never occurs in a fundus photograph, so the heat cannot be mistaken for a lesion
# (a white or yellow core could pass for an exudate). Powder blue rising to cobalt.
_HEAT = _lut([(0.0, "#8DB9F0"), (1.0, "#2350B8")])


def palette_overlay(img: np.ndarray, cam: np.ndarray, alpha: float = 0.62) -> np.ndarray:
    """Grad-CAM over the image in the app palette, with a contour at half the peak."""
    if cam.shape[:2] != img.shape[:2]:
        cam = cv2.resize(cam.astype(np.float32), (img.shape[1], img.shape[0]), interpolation=cv2.INTER_CUBIC)
    cam = np.clip(cam, 0, 1)
    colour = _HEAT[(cam * 255).astype(np.uint8)]
    w = (alpha * np.clip((cam - 0.15) / 0.85, 0, 1) ** 0.8)[..., None]
    out = np.clip(img * (1 - w) + colour * w, 0, 255).astype(np.uint8)
    contours, _ = cv2.findContours((cam >= 0.5).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, contours, -1, (233, 232, 248), max(1, img.shape[0] // 256), lineType=cv2.LINE_AA)
    return out

In [ ]:
%%writefile fundalux/export.py
"""Export a bundle's PyTorch weights for ONNX Runtime serving (needs PyTorch).

    python -m fundalux.export [model_dir]

Writes <model_dir>/onnx/: one file per DR network with two outputs (feature
map and logits, for the closed-form Grad-CAM in inference.py), the quality
gate, and heads.npz with the classifier parameters the heatmaps need. Records
the export under "onnx" in config.json. The server then needs no PyTorch.
"""

from __future__ import annotations

import inspect
import json
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn

from .inference import HEADS_FILE, read_config
from .models import FundusNet, load_weights

OPSET = 17


class _FeaturesAndLogits(nn.Module):
    def __init__(self, model: FundusNet):
        super().__init__()
        self.model = model

    def forward(self, x):
        feats = self.model.forward_features(x)
        return feats, self.model.forward_head(feats)


class _Logits(nn.Module):
    def __init__(self, model: FundusNet):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)


def _export(module: nn.Module, size: int, path: Path, outputs: list[str]) -> None:
    args = (module.eval(), torch.zeros(1, 3, size, size), str(path))
    try:  # the classic exporter: dynamic batch, no extra packages
        legacy = {"dynamo": False} if "dynamo" in inspect.signature(torch.onnx.export).parameters else {}
        torch.onnx.export(*args, input_names=["image"], output_names=outputs, dynamic_axes={"image": {0: "batch"}, **{o: {0: "batch"} for o in outputs}}, opset_version=OPSET, **legacy)
    except Exception as exc:  # newer PyTorch: the default exporter (batch of one, which is all the server uses)
        print(f"  classic ONNX exporter failed ({type(exc).__name__}); using the default exporter")
        torch.onnx.export(*args, input_names=["image"], output_names=outputs)


def head_parameters(model: FundusNet) -> dict:
    """What the closed-form Grad-CAM needs from the classifier head."""
    norm = model.norm
    layernorm = isinstance(norm, nn.Module) and hasattr(norm, "normalized_shape")
    channels = model.head.in_features
    return {
        "weight": model.head.weight.detach().cpu().numpy().astype(np.float32),
        "kind": "layernorm" if layernorm else "linear",
        "gamma": (norm.weight.detach().cpu().numpy() if layernorm else np.ones(channels)).astype(np.float32),
        "eps": float(norm.eps) if layernorm else 0.0,
    }


def export_bundle(model_dir, members: list[tuple[str, FundusNet, int]] | None = None, quality: FundusNet | None = None, quality_size: int | None = None) -> dict:
    """Export the bundle in `model_dir` (or the given in-memory models) and update its config.json."""
    model_dir = Path(model_dir)
    config = read_config(model_dir)
    dr = config["dr"]
    if members is None:
        specs = dr.get("models") or [{"arch": dr["arch"], "weights": dr["weights"], "input_size": config["image_size"]}]
        members = [(s["arch"], load_weights(FundusNet(s["arch"], num_grades=dr["num_grades"]), model_dir / s["weights"]), s["input_size"]) for s in specs]
    if quality is None:
        quality = load_weights(FundusNet(config["quality"]["arch"], num_grades=0), model_dir / config["quality"]["weights"])
    quality_size = quality_size or config["quality"]["input_size"]

    out = model_dir / "onnx"
    out.mkdir(exist_ok=True)
    heads, spec = {}, {"dir": "onnx", "opset": OPSET, "members": [], "heads": HEADS_FILE}
    for i, (arch, model, size) in enumerate(members):
        model = model.cpu().eval()
        name = f"dr_{i + 1}_{arch}.onnx"
        _export(_FeaturesAndLogits(model), size, out / name, ["features", "logits"])
        for k, v in head_parameters(model).items():
            heads[f"m{i}_{k}"] = np.asarray(v)
        spec["members"].append({"arch": arch, "file": name, "input_size": int(size)})
    _export(_Logits(quality.cpu().eval()), quality_size, out / "quality.onnx", ["logits"])
    spec["quality"] = "quality.onnx"
    np.savez(out / HEADS_FILE, **heads)
    config["onnx"] = spec
    (model_dir / "config.json").write_text(json.dumps(config, indent=2))
    return spec


def main() -> None:
    model_dir = Path(sys.argv[1]) if len(sys.argv) > 1 else Path("models")
    spec = export_bundle(model_dir)
    size = sum((model_dir / spec["dir"] / f).stat().st_size for f in [m["file"] for m in spec["members"]] + [spec["quality"], spec["heads"]])
    print(f"Exported {len(spec['members'])} networks and the quality gate to {model_dir / spec['dir']} ({size / 1e6:.0f} MB)")


if __name__ == "__main__":
    main()

In [ ]:
import fundalux
from fundalux import GRADE_NAMES, REFERABLE_MIN_GRADE
from fundalux import datasets as D
from fundalux import metrics as M
from fundalux.data import DRDataset, QualityDataset, worker_init
from fundalux.degrade import KINDS, degrade
from fundalux.gradcam import ensemble_grad_cam, lesion_overlap, overlay_heatmap
from fundalux.models import FundusNet, ensemble_probabilities, load_weights, save_weights, to_tensor
from fundalux.export import export_bundle
from fundalux.pipeline import Fundalux
from fundalux.sites import SiteRegistry, builtin_profiles, model_fingerprint
from fundalux.preprocess import correct_illumination, crop_fundus, enhance_fundus, model_input, read_image, retina_mask

print("fundalux", fundalux.__version__)

## 3. Storage

* `/kaggle/working` — saved with the notebook: the result zip and the training checkpoints.
* `/kaggle/temp` — discarded when the session ends: the cropped-image cache (about 2 GB).

A checkpoint is written after every epoch. If an *interactive* session is interrupted, re-running the notebook
continues from the last finished epoch, provided *Session options → Persistence* is set to **Files only**.
A committed run (*Save & Run All*) is not affected by the browser at all.

In [ ]:
RUN = "+".join(f"{m['arch']}@{m['size']}" for m in MODELS) + ("_balanced" if BALANCE_SOURCES else "") + (f"_sub{MAX_EYEPACS}" if MAX_EYEPACS else "") + ("_smoke" if SMOKE_TEST else "")
CACHE_TAG = f"cache_{IMG_SIZE}" + (f"_sub{MAX_EYEPACS}" if MAX_EYEPACS else "") + ("_smoke" if SMOKE_TEST else "")
DATA_DIR, CACHE_DIR, OUT_DIR = SCRATCH / "data", SCRATCH / CACHE_TAG, SCRATCH / "fundalux_outputs"
CKPT_DIR = WORK / "checkpoints"
FIG_DIR, REPORT_DIR = OUT_DIR / "figures", OUT_DIR / "reports"
for folder in (DATA_DIR, CKPT_DIR, FIG_DIR, REPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
INDEX_CSV, SEG_CSV = CACHE_DIR / "index.csv", CACHE_DIR / "idrid_seg.csv"
print(f"Results and checkpoints: {WORK}\nTemporary files:         {SCRATCH}")

metrics = {"run": RUN, "smoke_test": SMOKE_TEST, "image_size": IMG_SIZE}  # everything measured below ends up in metrics.json


def show(fig, name):
    """Save a figure into the output bundle and display it."""
    fig.savefig(FIG_DIR / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 4. Data

| Dataset | Add this Kaggle dataset as Input | Role | Images |
|---|---|---|---|
| EyePACS (Kaggle 2015, resized) | `tanlikesmath/diabetic-retinopathy-resized` | training / validation / test | ~35,000 |
| APTOS 2019 (Aravind Eye Hospital, India) | `mariaherrerot/aptos2019` | training / validation / test | ~3,660 |
| IDRiD (India, with lesion masks) | `aaryapatel98/indian-diabetic-retinopathy-image-dataset` | **external test** + Grad-CAM check — never trained on | 516 + 81 |
| Messidor-2 (France) | `mariaherrerot/messidor2preprocess` | **external test** — never trained on | 1,744 |
| EyeQ | fetched from GitHub | quality labels (good / usable / reject) for 12,543 EyePACS images | labels only |

Each dataset is looked for in this order: `LOCAL_DATA` (your own upload) → the notebook's Inputs
(`/kaggle/input`) → attached or downloaded automatically.
Every image is cropped to the retina once and stored at `IMG_SIZE`; training then reads the small copies.

In [ ]:
def fetch_dataset(name):
    """Return a folder holding the dataset: your own copy, a notebook Input, else attach/download it."""
    if name in LOCAL_DATA:
        return LOCAL_DATA[name]
    slug = DATASETS[name]
    mounted = D.find_kaggle_input(slug)
    if mounted:
        return mounted
    try:
        import kagglehub

        return kagglehub.dataset_download(slug)  # inside a Kaggle notebook this attaches the dataset
    except Exception as exc:
        print(f"kagglehub could not fetch {slug} ({type(exc).__name__}: {exc}); trying a direct download.")
    dest, archive = DATA_DIR / name, DATA_DIR / f"{name}.zip"
    url = f"https://www.kaggle.com/api/v1/datasets/download/{slug}"
    subprocess.run(["curl", "-L", "--fail", "--retry", "3", "-sS", "-o", str(archive), url], check=True)
    subprocess.run(["unzip", "-q", "-o", str(archive), "-d", str(dest)], check=True)
    archive.unlink()
    return str(dest)


def save_raw_examples(index, n=4):
    """Keep a few small raw photographs so the preprocessing figure can show the 'before'."""
    (CACHE_DIR / "examples").mkdir(parents=True, exist_ok=True)
    picks = index.groupby("source").head(1).head(n)
    for i, path in enumerate(picks["path"]):
        raw = read_image(path)
        scale = 512 / max(raw.shape[:2])
        raw = cv2.resize(raw, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA) if scale < 1 else raw
        cv2.imwrite(str(CACHE_DIR / "examples" / f"raw_{i}.jpg"), cv2.cvtColor(raw, cv2.COLOR_RGB2BGR))


def prepare_data():
    t0 = time.time()
    if SMOKE_TEST:
        from fundalux.synthetic import make_fake_datasets

        roots = make_fake_datasets(DATA_DIR / "fake", seed=SEED)
        eyeq_csv = roots["eyeq_csv"]
    else:
        roots = {}
        for name in DATASETS:
            roots[name] = fetch_dataset(name)
            print(f"{name}: {roots[name]}  ({time.time() - t0:.0f}s)")
        eyeq_csv = DATA_DIR / "Label_EyeQ_train.csv"
        try:
            urllib.request.urlretrieve(EYEQ_URL, eyeq_csv)
        except Exception as exc:
            print(f"EyeQ quality labels could not be downloaded ({exc}). The quality gate will learn from synthetic degradations only.")

    index = pd.concat([D.INDEXERS[name](roots[name]) for name in D.INDEXERS], ignore_index=True)
    if MAX_EYEPACS:
        patients = index.loc[index.source == "eyepacs", "patient"].drop_duplicates()
        keep = set(patients.sample(min(len(patients), MAX_EYEPACS // 2), random_state=SEED))
        index = index[(index.source != "eyepacs") | index.patient.isin(keep)]
    index = D.add_eyeq_labels(index, [eyeq_csv])
    held_out = 0.25 if SMOKE_TEST else 0.10
    index = D.assign_splits(index, seed=SEED, val_frac=held_out, test_frac=held_out)

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    save_raw_examples(index)
    print(f"Cropping {len(index)} images to {IMG_SIZE}px ...")
    index = D.build_cache(index, CACHE_DIR, IMG_SIZE, progress=tqdm)
    seg = D.cache_idrid_masks(D.index_idrid_segmentation(roots["idrid"]), CACHE_DIR, IMG_SIZE, progress=tqdm)
    seg.to_csv(SEG_CSV, index=False)
    index.drop(columns=["path"]).to_csv(INDEX_CSV, index=False)  # written last: its presence marks a complete cache
    print(f"Cache ready in {(time.time() - t0) / 60:.1f} min")


if not INDEX_CSV.exists():
    prepare_data()

index = pd.read_csv(INDEX_CSV, dtype={"image_id": str, "patient": str})
index["file"] = [str(CACHE_DIR / p) for p in index["cache_path"]]
seg = pd.read_csv(SEG_CSV)
has_grade = index.grade >= 0

print(f"\n{len(index)} images. Images per source and split:")
display(index.groupby(["source", "split"]).agg(images=("grade", "size"), referable=("referable", "sum")).unstack("split").fillna(0).astype(int))
print("DR grades per source:")
display(index[has_grade].groupby(["source", "grade"]).size().unstack("grade").fillna(0).astype(int).rename(columns=dict(enumerate(GRADE_NAMES))))
print("EyeQ quality labels (0 good, 1 usable, 2 reject):", index.eyeq.value_counts().sort_index().to_dict())
print("No retina found in", int((~index.fundus_found).sum()), "images (kept; the quality gate must reject them).")
metrics["data"] = {
    "images": {f"{s}/{p}": int(n) for (s, p), n in index.groupby(["source", "split"]).size().items()},
    "eyeq_labelled": int(index.eyeq.notna().sum()),
    "idrid_lesion_mask_images": int(len(seg)),
}

### Preprocessing, step by step

Crop to the retina → illumination correction → CLAHE. The second figure shows the simulated field problems
(blur, motion, glare, haze, under-exposure) used for augmentation and for testing the quality gate.

In [ ]:
examples = sorted((CACHE_DIR / "examples").glob("raw_*.jpg"))
picks = index.groupby("source").head(1).head(len(examples)) if examples else index.groupby("source").head(1)
fig, axes = plt.subplots(len(picks), 4, figsize=(11, 2.8 * len(picks)), squeeze=False)
for row, (_, r) in enumerate(picks.iterrows()):
    cropped = read_image(r.file)
    flat = correct_illumination(cropped)
    stages = [read_image(examples[row]) if examples else cropped, cropped, flat, enhance_fundus(cropped)]
    for ax, img, title in zip(axes[row], stages, ["original", "cropped", "+ illumination correction", "+ CLAHE (model input)"]):
        ax.imshow(img)
        ax.set_title(f"{r.source}: {title}" if title == "original" else title, fontsize=9)
        ax.axis("off")
show(fig, "preprocessing_steps")

sample = read_image(index[index.split == "test"].file.iloc[0])
levels = [0.15, 0.3, 0.5, 0.7, 1.0]
fig, axes = plt.subplots(len(KINDS), len(levels), figsize=(10, 2 * len(KINDS)), squeeze=False)
for i, kind in enumerate(KINDS):
    for j, severity in enumerate(levels):
        axes[i, j].imshow(degrade(sample, kind, severity, np.random.default_rng(SEED)))
        axes[i, j].set_title(f"{kind}  severity {severity}", fontsize=8)
        axes[i, j].axis("off")
show(fig, "field_degradations")

## 5. Training tools

One training loop serves every model. It fine-tunes the whole network with AdamW, a warm-up + cosine
learning-rate schedule and mixed precision, keeps the weights with the best validation score, and saves a
checkpoint after every epoch so an interrupted run resumes.

Two safety nets for a long unattended run: if a model does not fit in GPU memory its batch size is halved and
it starts again, and training stops early rather than run past `TIME_BUDGET_HOURS`.

In [ ]:
NUM_WORKERS = 0 if SMOKE_TEST else min(4, os.cpu_count() or 2)


def make_loader(ds, batch_size, shuffle=False, sampler=None):
    training = shuffle or sampler is not None
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle and sampler is None, sampler=sampler, drop_last=training, num_workers=NUM_WORKERS, pin_memory=USE_AMP, worker_init_fn=worker_init)


@torch.no_grad()
def predict(model, ds, batch_size, desc="predicting"):
    """Run `model` over a dataset. Returns (logits [N, outputs], targets [N]) as numpy arrays."""
    model.eval()
    outputs, targets = [np.zeros((0, model.head.out_features), np.float32)], [np.zeros(0)]
    for x, y in tqdm(make_loader(ds, batch_size), desc=desc, leave=False):
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            out = model(x.to(DEVICE, non_blocking=True))
        outputs.append(out.float().cpu().numpy())
        targets.append(y.numpy())
    return np.concatenate(outputs), np.concatenate(targets)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.asarray(z, dtype=np.float64)))


def make_scaler():
    if hasattr(torch.amp, "GradScaler"):
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    return torch.cuda.amp.GradScaler(enabled=USE_AMP)  # torch < 2.3


def lr_at(step, total_steps, peak):
    """Linear warm-up over the first 10% of training, then cosine decay to 1% of the peak."""
    warmup = max(1, total_steps // 10)
    if step < warmup:
        return peak * (step + 1) / warmup
    progress = (step - warmup) / max(1, total_steps - warmup)
    return peak * (0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * progress)))


def fit(model, train_ds, val_ds, loss_fn, score_fn, epochs, lr, batch_size, tag, sampler=None, deadline=None):
    """Train `model`, leave it holding its best-validation weights and return the per-epoch history.

    `sampler` replaces plain shuffling; `deadline` (a time.time() value) stops training
    before an epoch that would not finish in time.
    """
    last_path, best_path = CKPT_DIR / f"{tag}_last.pt", CKPT_DIR / f"{tag}_best.pt"
    loader = make_loader(train_ds, batch_size, shuffle=True, sampler=sampler)
    total_steps = epochs * len(loader)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scaler = make_scaler()
    state = {"epoch": 0, "best": -math.inf, "history": []}

    if last_path.exists() and not FRESH_START:
        ckpt = torch.load(last_path, map_location="cpu", weights_only=False)
        if ckpt["total_steps"] == total_steps:
            model.load_state_dict(ckpt["model"])
            opt.load_state_dict(ckpt["opt"])
            scaler.load_state_dict(ckpt["scaler"])
            state = ckpt["state"]
            print(f"[{tag}] resuming after epoch {state['epoch']} of {epochs}")
        else:
            print(f"[{tag}] saved checkpoint belongs to different settings - starting again")
    if state["epoch"] == 0:
        best_path.unlink(missing_ok=True)

    epoch_seconds = None
    for epoch in range(state["epoch"], epochs):
        if deadline and epoch_seconds and time.time() + 1.15 * epoch_seconds > deadline:
            print(f"[{tag}] stopping after {epoch} of {epochs} epochs to stay inside TIME_BUDGET_HOURS")
            break
        model.train()
        t0, loss_sum, seen = time.time(), 0.0, 0
        bar = tqdm(loader, desc=f"[{tag}] epoch {epoch + 1}/{epochs}", leave=False)
        for step, (x, y) in enumerate(bar, start=epoch * len(loader)):
            for group in opt.param_groups:
                group["lr"] = lr_at(step, total_steps, lr)
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(DEVICE.type, enabled=USE_AMP):
                loss = loss_fn(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt)
            scaler.update()
            loss_sum, seen = loss_sum + loss.item() * len(x), seen + len(x)
            bar.set_postfix(loss=f"{loss_sum / seen:.4f}")
        if not math.isfinite(loss_sum):  # diverged: keep the best earlier epoch if there is one
            print(f"[{tag}] the training loss became NaN/inf in epoch {epoch + 1} - stopping this model here")
            if not best_path.exists():
                raise FloatingPointError(f"[{tag}] diverged in its first epoch; lower this model's learning rate")
            break
        score = float(score_fn(*predict(model, val_ds, batch_size * 2, desc="validating")))
        epoch_seconds = time.time() - t0
        state["history"].append({"epoch": epoch + 1, "train_loss": loss_sum / seen, "val_score": score, "minutes": epoch_seconds / 60})
        state["epoch"] = epoch + 1
        # NaN (a validation set with one class, only in the smoke test) never counts as an improvement.
        if score > state["best"] or not best_path.exists():
            state["best"] = score if math.isfinite(score) else state["best"]
            save_weights(model, best_path)
        ckpt = {"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(), "state": state, "total_steps": total_steps}
        torch.save(ckpt, str(last_path) + ".tmp")
        os.replace(str(last_path) + ".tmp", last_path)
        print(f"[{tag}] epoch {epoch + 1}/{epochs}  train loss {loss_sum / seen:.4f}  val score {score:.4f}  ({epoch_seconds / 60:.1f} min)")

    load_weights(model, best_path).to(DEVICE)
    return state["history"]


OUT_OF_MEMORY = getattr(torch, "OutOfMemoryError", None) or torch.cuda.OutOfMemoryError


def fit_with_memory_fallback(make_model, batch_size, **fit_args):
    """Build a model and train it; on a GPU out-of-memory error halve the batch size and start that model again.

    Returns (model, history, batch size actually used).
    """
    while True:
        model, out_of_memory = make_model().to(DEVICE), False
        try:
            return model, fit(model, batch_size=batch_size, **fit_args), batch_size
        except OUT_OF_MEMORY:
            out_of_memory = True
        if out_of_memory:  # outside the except block, so the failed batch is no longer referenced
            del model
            gc.collect()
            torch.cuda.empty_cache()
            if batch_size <= 4:
                raise RuntimeError(f"[{fit_args['tag']}] does not fit in GPU memory even with batch size {batch_size}")
            batch_size //= 2
            print(f"[{fit_args['tag']}] out of GPU memory - retrying with batch size {batch_size}")


def plot_histories(histories, title, score_name):
    """`histories` maps a model name to its per-epoch history."""
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.2))
    for name, history in histories.items():
        h = pd.DataFrame(history)
        a.plot(h.epoch, h.train_loss, marker="o", label=name)
        b.plot(h.epoch, h.val_score, marker="o", label=name)
    a.set(title=f"{title}: training loss", xlabel="epoch")
    b.set(title=f"{title}: validation {score_name}", xlabel="epoch")
    b.legend(fontsize=8)
    return fig

## 6. Image-quality gate

A small network scores each photograph for "ungradable". It learns from two kinds of examples:

* **real** — EyeQ's human labels on EyePACS images (*reject* = ungradable, *good*/*usable* = gradable);
* **synthetic** — good images that we deliberately ruin with heavy blur, glare, haze or under-exposure.
  Mildly degraded copies stay labelled gradable, so the gate does not reject everything imperfect.

The gate sees the cropped image *without* CLAHE: enhancement would hide the defects it must detect.

In [ ]:
pool = index[index.split.isin(["train", "val", "test"])]
labelled = pool[pool.eyeq.notna()]
if len(labelled):
    extra = pool[(pool.source != "eyepacs")]  # other cameras, assumed gradable: they teach the gate not to reject a camera type
    q_frame = pd.concat([labelled.assign(q_label=(labelled.eyeq == 2).astype(float), q_clean=labelled.eyeq == 0), extra.assign(q_label=0.0, q_clean=True)])
else:
    print("No EyeQ labels: training the gate on synthetic degradations only.")
    q_frame = pool.assign(q_label=0.0, q_clean=True)
metrics["quality_gate"] = {"real_labels": int(len(labelled)), "real_ungradable": int((labelled.eyeq == 2).sum())}


def quality_ds(frame, train=False, synthetic=True, fixed_degradation=None):
    p = 0.25 if synthetic else 0.0
    return QualityDataset(frame.file, frame.q_label, frame.q_clean, IMG_SIZE, QUALITY_SIZE, train=train, severe_p=p, mild_p=p, seed=SEED, fixed_degradation=fixed_degradation,
                          gradable_range=GATE_GRADABLE_DEGRADATION, ungradable_range=GATE_UNGRADABLE_DEGRADATION)


def bce(out, y):
    return F.binary_cross_entropy_with_logits(out[:, 0].float(), y.float())


q_train, q_val, q_test = (q_frame[q_frame.split == s] for s in ("train", "val", "test"))
print(f"Quality gate: {len(q_train)} train / {len(q_val)} val / {len(q_test)} test images")
quality_model, history, _ = fit_with_memory_fallback(
    lambda: FundusNet(QUALITY_ARCH, num_grades=0, pretrained=not SMOKE_TEST), QUALITY_BATCH,
    train_ds=quality_ds(q_train, train=True), val_ds=quality_ds(q_val), loss_fn=bce, score_fn=lambda out, y: M.roc_auc(y, out[:, 0]),
    epochs=QUALITY_EPOCHS, lr=QUALITY_LR, tag=f"quality_{QUALITY_ARCH}_{QUALITY_SIZE}_{CACHE_TAG}",
)
show(plot_histories({QUALITY_ARCH: history}, "Quality gate", "AUC"), "quality_training")

In [ ]:
# Threshold: the point on the validation ROC that best separates gradable from ungradable (Youden's J).
out, y = predict(quality_model, quality_ds(q_val), QUALITY_BATCH * 2)
QUALITY_THRESHOLD = M.youden_threshold(y, sigmoid(out[:, 0]))
if not 0.0 < QUALITY_THRESHOLD < 1.0:
    QUALITY_THRESHOLD = 0.5
print(f"Quality threshold: reject when P(ungradable) >= {QUALITY_THRESHOLD:.3f}\n")

rows = {}
for label, frame, synthetic in [("test: real + synthetic", q_test, True), ("test: real EyeQ labels only", q_test[q_test.eyeq.notna()], False)]:
    if len(frame):
        out, y = predict(quality_model, quality_ds(frame, synthetic=synthetic), QUALITY_BATCH * 2)
        rep = M.screening_report(y, sigmoid(out[:, 0]), QUALITY_THRESHOLD, n_boot=0 if SMOKE_TEST else 500)
        rows[label] = {"images": rep["n"], "ungradable": rep["n_positive"], "AUC": rep["auc"], "ungradable caught": rep["sensitivity"], "gradable accepted": rep["specificity"]}
display(pd.DataFrame(rows).T.round(3))
metrics["quality_gate"].update(threshold=QUALITY_THRESHOLD, test=rows)

# Score every image once. The cached rule checks (retina found, enough retina visible, red fundus) are
# part of the gate too, exactly as in the app.
out, _ = predict(quality_model, QualityDataset(index.file, np.zeros(len(index)), np.zeros(len(index), bool), IMG_SIZE, QUALITY_SIZE, severe_p=0, mild_p=0), QUALITY_BATCH * 2, desc="scoring all images")
index["q_prob"] = sigmoid(out[:, 0])
if SMOKE_TEST:  # an untrained gate accepts or rejects everything; keep both branches alive for the code test
    QUALITY_THRESHOLD = float(index.q_prob.quantile(0.7))
    metrics["quality_gate"]["threshold"] = QUALITY_THRESHOLD
index["rule_pass"] = index.fundus_found & (index.retina_fraction >= 0.5) & (index.red_minus_blue > 0)
index["gate_pass"] = index.rule_pass & (index.q_prob < QUALITY_THRESHOLD)

held_out = index[index.split.isin(["test", "external"])]
by_source = held_out.groupby("source").agg(images=("gate_pass", "size"), rejected=("gate_pass", lambda s: 1 - s.mean()))
by_grade = held_out[held_out.grade >= 0].groupby("grade").agg(images=("gate_pass", "size"), rejected=("gate_pass", lambda s: 1 - s.mean())).rename(index=dict(enumerate(GRADE_NAMES)))
print("Share of held-out images the gate rejects. IDRiD and Messidor-2 are curated, gradable sets, so their rejections are false alarms.")
display(by_source.round(3))
print("Rejections by DR grade. A gate that rejects diseased eyes more often is safe only because a rejected patient is re-imaged or referred.")
display(by_grade.round(3))
metrics["quality_gate"].update(reject_rate_by_source=by_source.rejected.to_dict(), reject_rate_by_grade=by_grade.rejected.to_dict())

# What do good images look like? The app uses these ranges to tell the operator *why* an image was rejected.
good = index[index.eyeq == 0] if (index.eyeq == 0).sum() >= 20 else index[index.gate_pass]
HINT_RANGES = {k: [float(good[k].quantile(0.02)), float(good[k].quantile(0.98))] for k in ("sharpness", "brightness", "contrast")}
HINT_RANGES["saturated_fraction"] = [0.0, float(max(good.saturated_fraction.quantile(0.99), 0.01))]
print("Statistics of good images (2nd-98th percentile):", {k: [round(v, 2) for v in r] for k, r in HINT_RANGES.items()})

## 7. DR classifier: an ensemble of three networks

Each member is an ImageNet-pretrained backbone with one small head: output 0 is the **referable DR** logit,
outputs 1–5 are the five DR grades. The three members (`MODELS` in the Settings) have different designs and
see the image at different sizes; at prediction time their probabilities are averaged. Their mistakes are
partly different, so the average is usually better and steadier than any single member.

**Balanced sampling.** In the first training run the model's scores sat on a different scale for each camera,
and one threshold did not transfer. A likely cause: the sources differ in how much disease they contain
(about 20% referable in EyePACS, 41% in APTOS), so "which camera took this?" becomes a cheap hint about the
answer. With `BALANCE_SOURCES` every source is sampled so that half of its training images are referable:
the source no longer says anything about the label. Each source keeps its natural share of the batches.

Images that EyeQ's graders rejected are left out of training (their DR labels are unreliable); validation and
test keep every image. Training images are augmented with flips, rotation, colour jitter and *mild* field
degradations.

In [ ]:
dr_train = index[(index.split == "train") & (index.eyeq != 2)]
dr_val = index[index.split == "val"]
if BALANCE_SOURCES:
    # Weight n_source / (classes * n_cell): each source keeps its share, and inside it both classes get half.
    in_source = dr_train.groupby("source")["grade"].transform("size")
    in_cell = dr_train.groupby(["source", "referable"])["grade"].transform("size")
    classes = dr_train.groupby("source")["referable"].transform("nunique")
    sample_weight = (in_source / (classes * in_cell)).to_numpy(dtype=np.float64)
    pos_weight = 1.0
else:
    sample_weight = np.ones(len(dr_train))
    n_pos = float(dr_train.referable.sum())
    pos_weight = (len(dr_train) - n_pos) / max(n_pos, 1.0)
# Grade weights follow the grade mix the sampler will actually produce.
seen_grades = np.bincount(dr_train.grade, weights=sample_weight, minlength=5)
grade_weights = np.clip(seen_grades.sum() / (5 * np.maximum(seen_grades, 1e-9)), 0.5, 5.0)
per_source = dr_train.assign(w=sample_weight, wr=sample_weight * dr_train.referable).groupby("source").agg(images=("w", "size"), data=("referable", "mean"), w=("w", "sum"), wr=("wr", "sum"))
mix = pd.DataFrame({"images": per_source.images, "referable in the data": per_source.data, "referable as sampled": per_source.wr / per_source.w, "share of each batch": per_source.w / per_source.w.sum()})
print(f"DR classifier: {len(dr_train)} train / {len(dr_val)} val images")
display(mix.round(3))
print("Loss weights  grades:", grade_weights.round(2), " referable:", round(pos_weight, 2))
GRADE_W = torch.tensor(grade_weights, dtype=torch.float32, device=DEVICE)
POS_W = torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE)


def dr_loss(out, grade):
    referable = (grade >= REFERABLE_MIN_GRADE).float()
    binary = F.binary_cross_entropy_with_logits(out[:, 0].float(), referable, pos_weight=POS_W)
    grading = F.cross_entropy(out[:, 1:].float(), grade, weight=GRADE_W, label_smoothing=0.05)
    return binary + 0.5 * grading


def dr_ds(frame, size, train=False, fixed_degradation=None):
    return DRDataset(frame.file, frame.grade.clip(lower=0), size, train=train, seed=SEED, fixed_degradation=fixed_degradation)


def train_sampler():
    if not BALANCE_SOURCES:
        return None
    return torch.utils.data.WeightedRandomSampler(torch.as_tensor(sample_weight), num_samples=len(dr_train), replacement=True)


RESERVE_HOURS = 1.0  # kept back for evaluation, the Grad-CAM check, the robustness sweep and the export
members = []         # one dict per trained model: arch, size, batch, model, history
for i, spec in enumerate(MODELS):
    now = time.time()
    share = (RUN_START + (TIME_BUDGET_HOURS - RESERVE_HOURS) * 3600 - now) / (len(MODELS) - i)  # this model's slice of the remaining time
    if members and share < 20 * 60:
        print(f"SKIPPING {spec['arch']}: only {share / 60:.0f} min of the time budget left for it. The ensemble will have {len(members)} member(s).")
        continue
    tag = f"dr_{spec['arch']}_{spec['size']}_{CACHE_TAG}" + ("_balanced" if BALANCE_SOURCES else "")
    try:
        model, history, batch = fit_with_memory_fallback(
            lambda: FundusNet(spec["arch"], num_grades=5, pretrained=not SMOKE_TEST), spec["batch"],
            train_ds=dr_ds(dr_train, spec["size"], train=True), val_ds=dr_ds(dr_val, spec["size"]), loss_fn=dr_loss,
            score_fn=lambda out, g: M.roc_auc(g >= REFERABLE_MIN_GRADE, out[:, 0]), epochs=EPOCHS, lr=spec["lr"], tag=tag,
            sampler=train_sampler(), deadline=now + share,
        )
    except FloatingPointError as exc:  # one model failing must not cost the whole run
        print(f"SKIPPING {spec['arch']}: {exc}")
        continue
    members.append({"arch": spec["arch"], "size": spec["size"], "batch": batch, "model": model, "history": history})
    print(f"{spec['arch']} done: best validation AUC {max(h['val_score'] for h in history):.4f} in {sum(h['minutes'] for h in history) / 60:.2f} h ({len(history)} of {EPOCHS} epochs)\n")

if not members:
    raise RuntimeError("No DR model could be trained - see the messages above.")
if len(members) < len(MODELS):
    print(f"WARNING: the ensemble has {len(members)} of the {len(MODELS)} requested models.")
show(plot_histories({m["arch"]: m["history"] for m in members}, "DR classifier", "AUC (referable DR)"), "dr_training")
metrics["dr_training"] = {
    "epochs": EPOCHS, "train_images": int(len(dr_train)), "balanced_sources": BALANCE_SOURCES, "requested_models": [s["arch"] for s in MODELS],
    "members": [{k: m[k] for k in ("arch", "size", "batch", "history")} for m in members],
}

## 8. Decision threshold and results

The ensemble score is the average of the members' probabilities. The threshold is chosen **on validation data
only**: the highest threshold that still reaches `VAL_SENSITIVITY`. It is then frozen and applied to data the
models have never seen: the internal test split, and the two external datasets from other hospitals and cameras.

The headline numbers are for images the quality gate **accepts**, because those are the only images the
classifier grades in use. Results on *all* images are shown next to them.

In [ ]:
def ensemble_scores(frame, fixed_degradation=None, desc="scoring"):
    """Average the members. Returns (P(referable) [N], grade probabilities [N, 5], each member's P(referable) [members, N])."""
    logits = [predict(m["model"], dr_ds(frame, m["size"], fixed_degradation=fixed_degradation), m["batch"] * 2, desc=f"{desc}: {m['arch']}")[0] for m in members]
    return ensemble_probabilities(logits)


scored = index[index.split.isin(["val", "test", "external"]) & has_grade]
p_ref, p_grade, p_members = ensemble_scores(scored, desc="validation and test images")
index.loc[scored.index, "p_ref"] = p_ref
index.loc[scored.index, "grade_pred"] = p_grade.argmax(axis=1)
MEMBER_COLUMNS = [f"p_{m['arch']}" for m in members]
for column, p in zip(MEMBER_COLUMNS, p_members):
    index.loc[scored.index, column] = p

val = index[(index.split == "val") & has_grade & index.gate_pass]
DR_THRESHOLD = M.threshold_for_sensitivity(val.referable, val.p_ref, VAL_SENSITIVITY)
val_sens, val_spec = M.sens_spec(val.referable, val.p_ref, DR_THRESHOLD)
print(f"Threshold: refer when P(referable DR) >= {DR_THRESHOLD:.3f}   (validation: sensitivity {val_sens:.3f}, specificity {val_spec:.3f})")

EVAL_SETS = {
    "Internal test (EyePACS + APTOS)": index[(index.split == "test") & has_grade],
    "  EyePACS test": index[(index.split == "test") & (index.source == "eyepacs")],
    "  APTOS test": index[(index.split == "test") & (index.source == "aptos")],
    "IDRiD - external, India": index[(index.source == "idrid") & has_grade],
    "Messidor-2 - external": index[(index.source == "messidor") & has_grade],
}
N_BOOT = 50 if SMOKE_TEST else 1000
results, table, member_auc = {}, {}, {}
for name, frame in EVAL_SETS.items():
    accepted = frame[frame.gate_pass]
    rep = M.screening_report(accepted.referable, accepted.p_ref, DR_THRESHOLD, n_boot=N_BOOT, seed=SEED)
    rep_all = M.screening_report(frame.referable, frame.p_ref, DR_THRESHOLD, n_boot=0)
    rep["rejected_by_gate"] = 1 - len(accepted) / max(len(frame), 1)
    rep["grade_kappa"] = M.quadratic_weighted_kappa(accepted.grade, accepted.grade_pred, 5)
    rep["all_images"] = rep_all
    results[name.strip()] = rep
    member_auc[name.strip()] = {**{m["arch"]: M.roc_auc(accepted.referable, accepted[c]) for m, c in zip(members, MEMBER_COLUMNS)}, "ensemble": rep["auc"]}
    ci = lambda key: "[{:.3f}, {:.3f}]".format(*rep[key + "_ci"])
    table[name] = {
        "graded": rep["n"], "referable": rep["n_positive"], "rejected": f"{rep['rejected_by_gate']:.1%}",
        "AUC": round(rep["auc"], 3), "sensitivity": round(rep["sensitivity"], 3), "sens. 95% CI": ci("sensitivity"),
        "specificity": round(rep["specificity"], 3), "spec. 95% CI": ci("specificity"), "grade kappa": round(rep["grade_kappa"], 3),
        "sens. (all images)": round(rep_all["sensitivity"], 3), "spec. (all images)": round(rep_all["specificity"], 3),
    }
display(pd.DataFrame(table).T)
print("AUC of each member on its own, and of the ensemble (gate-accepted images):")
display(pd.DataFrame(member_auc).T.round(4))
metrics["dr"] = {
    "threshold": DR_THRESHOLD, "validation": {"sensitivity": val_sens, "specificity": val_spec, "target_sensitivity": VAL_SENSITIVITY},
    "results": results, "member_auc": member_auc,
}

print(f"\nScreening targets: sensitivity > {TARGET_SENSITIVITY:.0%} and specificity > {TARGET_SPECIFICITY:.0%} on held-out data")
criteria = {}
for name in ("Internal test (EyePACS + APTOS)", "IDRiD - external, India", "Messidor-2 - external"):
    r = results[name]
    criteria[name] = bool(r["sensitivity"] > TARGET_SENSITIVITY and r["specificity"] > TARGET_SPECIFICITY)
    print(f"  {'MET    ' if criteria[name] else 'NOT MET'}  {name}: sensitivity {r['sensitivity']:.1%}, specificity {r['specificity']:.1%}")
metrics["success_criteria"] = {"target_sensitivity": TARGET_SENSITIVITY, "target_specificity": TARGET_SPECIFICITY, "met": criteria}
if SMOKE_TEST:
    print("  (smoke test on synthetic images: these numbers mean nothing)")

# Does one threshold transfer between cameras? For each set, the threshold that would give 90% sensitivity
# *on that set*. Chosen with the test labels, so this is a diagnosis, never a result: the closer these
# numbers are to each other, the better a single threshold works everywhere.
transfer = {}
for name, frame in EVAL_SETS.items():
    accepted = frame[frame.gate_pass]
    own = M.threshold_for_sensitivity(accepted.referable, accepted.p_ref, TARGET_SENSITIVITY)
    transfer[name] = {"threshold for 90% sensitivity on this set": own, "specificity there": M.sens_spec(accepted.referable, accepted.p_ref, own)[1]}
print(f"\nThreshold transfer check (the frozen threshold is {DR_THRESHOLD:.3f}):")
display(pd.DataFrame(transfer).T.round(3))
metrics["dr"]["threshold_transfer_diagnostic"] = {k.strip(): v for k, v in transfer.items()}

In [ ]:
# ROC curves with the chosen operating point, and the 5-grade confusion matrix on the internal test set.
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for name in ("Internal test (EyePACS + APTOS)", "IDRiD - external, India", "Messidor-2 - external"):
    frame = EVAL_SETS[name][EVAL_SETS[name].gate_pass]
    if frame.referable.nunique() < 2:
        continue
    fpr, tpr, _ = M.roc_curve(frame.referable, frame.p_ref)
    line, = axes[0].plot(fpr, tpr, label=f"{name}  (AUC {results[name]['auc']:.3f})")
    axes[0].plot(1 - results[name]["specificity"], results[name]["sensitivity"], "o", color=line.get_color(), markersize=8)
axes[0].plot([0, 1], [0, 1], ":", color="grey")
axes[0].axhline(TARGET_SENSITIVITY, ls="--", lw=0.8, color="grey")
axes[0].axvline(1 - TARGET_SPECIFICITY, ls="--", lw=0.8, color="grey")
axes[0].set(xlabel="1 - specificity", ylabel="sensitivity", title=f"Referable DR, ensemble: ROC (dots = threshold {DR_THRESHOLD:.2f}; dashed = targets)")
axes[0].legend(loc="lower right", fontsize=8)

frame = EVAL_SETS["Internal test (EyePACS + APTOS)"]
frame = frame[frame.gate_pass]
cm = M.confusion_matrix(frame.grade, frame.grade_pred, 5)
axes[1].imshow(cm / np.maximum(cm.sum(axis=1, keepdims=True), 1), cmap="Blues", vmin=0, vmax=1)
for i in range(5):
    for j in range(5):
        axes[1].text(j, i, cm[i, j], ha="center", va="center", fontsize=9, color="white" if cm[i, j] > 0.5 * max(cm[i].sum(), 1) else "black")
axes[1].set(xticks=range(5), yticks=range(5), xlabel="predicted grade", ylabel="true grade", title="5-grade confusion matrix (internal test)")
axes[1].set_xticklabels(GRADE_NAMES, rotation=30, ha="right", fontsize=8)
axes[1].set_yticklabels(GRADE_NAMES, fontsize=8)
show(fig, "roc_and_confusion")
# Validation rows are included so a different operating point can be studied later without retraining.
index[index.split.isin(["val", "test", "external"])][["source", "image_id", "split", "grade", "referable", "q_prob", "gate_pass", "p_ref", "grade_pred", *MEMBER_COLUMNS]].to_csv(REPORT_DIR / "predictions.csv", index=False)

## 9. Grad-CAM: where does the ensemble look?

Every member explains its own prediction; the three heatmaps are averaged. A heatmap that *looks* plausible is
not evidence. IDRiD provides pixel-level lesion masks drawn by ophthalmologists (microaneurysms, haemorrhages,
hard and soft exudates) for 81 images the models never trained on, so agreement can be measured:

* **pointing hit** — the hottest point of the heatmap lies on (or within 4% of the image width of) a lesion;
* **energy on lesions ÷ lesion area** — how many times more heat falls on lesions than a uniform map would put there (1 = chance);
* **pixel AUC** — the heatmap used as a lesion detector (0.5 = chance).

As a sanity check the same scores are computed for a **baseline copy** of the ensemble that has never seen a
DR label (ImageNet backbones, random heads). If the trained ensemble does not clearly beat it, the heatmaps are
showing generic image saliency, not learned evidence.

In [ ]:
def cams_for(models_and_sizes, files, batch=8):
    """Ensemble Grad-CAM (referable-DR output) for cached images. Returns (enhanced images, heatmaps at IMG_SIZE, P(referable))."""
    images, cams, probs = [], [], []
    for i in range(0, len(files), batch):
        cropped = [read_image(f) for f in files[i : i + batch]]
        cam, logits = ensemble_grad_cam(models_and_sizes, cropped, IMG_SIZE, DEVICE)
        images += [model_input(c, IMG_SIZE) for c in cropped]
        cams += list(cam)
        probs += ensemble_probabilities(logits)[0].tolist()
    return images, cams, probs


trained_members = [(m["model"], m["size"]) for m in members]
seg_files = [str(CACHE_DIR / p) for p in seg.cache_path]
seg_masks = [cv2.imread(str(CACHE_DIR / p), cv2.IMREAD_UNCHANGED) for p in seg.mask_path]
seg_images, seg_cams, seg_probs = cams_for(trained_members, seg_files)
torch.manual_seed(SEED)
baseline_members = [(FundusNet(m["arch"], num_grades=5, pretrained=not SMOKE_TEST).to(DEVICE).eval(), m["size"]) for m in members]
_, base_cams, _ = cams_for(baseline_members, seg_files)
del baseline_members

rows = []
for image_id, image, mask, cam, base, prob in zip(seg.image_id, seg_images, seg_masks, seg_cams, base_cams, seg_probs):
    lesion, valid = (mask & D.LESION_ANY) > 0, retina_mask(image) > 0
    trained, baseline = lesion_overlap(cam, lesion, valid), lesion_overlap(base, lesion, valid)
    rows.append({"image_id": image_id, "p_referable": prob, "flagged": prob >= DR_THRESHOLD, **trained, **{f"baseline_{k}": v for k, v in baseline.items() if k != "lesion_area"}})
cam_report = pd.DataFrame(rows)
cam_report.to_csv(REPORT_DIR / "gradcam_vs_idrid_lesions.csv", index=False)

for prefix in ("", "baseline_"):
    cam_report[prefix + "lift"] = cam_report[prefix + "energy_in_mask"] / cam_report["lesion_area"]
summary = pd.DataFrame({
    "trained model": [cam_report.pointing_hit.mean(), cam_report.lift.median(), cam_report.pixel_auc.mean()],
    "baseline (no DR training)": [cam_report.baseline_pointing_hit.mean(), cam_report.baseline_lift.median(), cam_report.baseline_pixel_auc.mean()],
    "chance": [float("nan"), 1.0, 0.5],
}, index=["pointing hit rate", "energy on lesions / lesion area (median)", "pixel AUC (mean)"])
display(summary.round(3))
paired = cam_report.dropna(subset=["pixel_auc", "baseline_pixel_auc"])
wins = int((paired.pixel_auc > paired.baseline_pixel_auc).sum())
auc_ci = M.bootstrap_ci(lambda _, v: float(np.mean(v)), np.zeros(len(paired)), paired.pixel_auc.to_numpy(), n_boot=N_BOOT, seed=SEED) if len(paired) else (float("nan"),) * 2
cam_ok = bool(len(paired) and auc_ci[0] > 0.5 and wins > len(paired) / 2)
print(f"{len(seg)} IDRiD images with lesion masks; the ensemble flags {int(cam_report.flagged.sum())} as referable; {int(cam_report['empty'].sum())} have an empty heatmap (counted as misses).")
print(f"Mean pixel AUC {paired.pixel_auc.mean():.3f}, 95% CI [{auc_ci[0]:.3f}, {auc_ci[1]:.3f}]; better than the baseline on {wins} of {len(paired)} images.")
print("Sanity check", "PASSED: heatmaps overlap lesions more than chance and more than networks with no DR training." if cam_ok else "FAILED: heatmaps do not reliably overlap the annotated lesions - do not present them as explanations.")
print("Note: Grad-CAM on these backbones has a coarse grid (about 1/32 of the image), so it marks regions, not individual microaneurysms.")
metrics["gradcam"] = {
    "images": int(len(seg)), "flagged_referable": int(cam_report.flagged.sum()), "empty_heatmaps": int(cam_report["empty"].sum()),
    "trained": summary["trained model"].to_dict(), "baseline": summary["baseline (no DR training)"].to_dict(),
    "pixel_auc_ci": list(auc_ci), "beats_baseline_on": wins, "compared": int(len(paired)), "sanity_check_passed": cam_ok,
}

In [ ]:
order = np.argsort(-cam_report.pixel_auc.fillna(-1).to_numpy())
picks = list(order[:2]) + list(order[len(order) // 2 : len(order) // 2 + 1]) + list(order[-1:])  # best, typical, worst
fig, axes = plt.subplots(len(picks), 3, figsize=(9, 3 * len(picks)), squeeze=False)
for row, i in enumerate(picks):
    outline = cv2.dilate(((seg_masks[i] & D.LESION_ANY) > 0).astype(np.uint8), np.ones((3, 3), np.uint8))
    marked = seg_images[i].copy()
    marked[outline > 0] = (0, 255, 0)
    r = cam_report.iloc[i]
    for ax, img, title in zip(axes[row], [seg_images[i], marked, overlay_heatmap(seg_images[i], seg_cams[i])], [f"{r.image_id}  P(referable) = {r.p_referable:.2f}", "ophthalmologists' lesion masks", f"Grad-CAM (pixel AUC {r.pixel_auc:.2f})"]):
        ax.imshow(img)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
show(fig, "gradcam_vs_lesion_masks")

# Heatmaps for ordinary test images: the most confident referable cases and the most confident normals.
test = index[(index.split == "test") & has_grade & index.gate_pass].sort_values("p_ref")
picks = pd.concat([test.tail(4), test.head(2)])
images, cams, _ = cams_for(trained_members, list(picks.file))
fig, axes = plt.subplots(2, len(picks), figsize=(2.6 * len(picks), 5.6), squeeze=False)
for col, (r, img, cam) in enumerate(zip(picks.itertuples(), images, cams)):
    axes[0, col].imshow(img)
    axes[0, col].set_title(f"true: {GRADE_NAMES[r.grade]}", fontsize=8)
    axes[1, col].imshow(overlay_heatmap(img, cam))
    axes[1, col].set_title(f"P(referable) = {r.p_ref:.2f}", fontsize=8)
    axes[0, col].axis("off")
    axes[1, col].axis("off")
show(fig, "gradcam_examples")

## 10. Field conditions: does the gate protect the classifier?

Good test images are degraded on purpose, at increasing severity. For each level we record how many images
the gate rejects, and the ensemble's sensitivity and specificity with and without the gate. The gate is doing
its job when it rejects images *before* the classifier starts to fail on them.

In [ ]:
clean = index[(index.split == "test") & has_grade & index.gate_pass]
clean = clean.sample(min(len(clean), 16 if SMOKE_TEST else 600), random_state=SEED)
clean = clean.assign(q_label=0.0, q_clean=True)
severities = [0.3, 0.7] if SMOKE_TEST else [0.15, 0.3, 0.5, 0.7, 1.0]
base_sens, base_spec = M.sens_spec(clean.referable, clean.p_ref, DR_THRESHOLD)
rows = [{"kind": k, "severity": 0.0, "rejected": 0.0, "sens_no_gate": base_sens, "spec_no_gate": base_spec, "sens_with_gate": base_sens, "spec_with_gate": base_spec} for k in KINDS]
for kind in KINDS:
    for severity in severities:
        q_out, _ = predict(quality_model, quality_ds(clean, synthetic=False, fixed_degradation=(kind, severity)), QUALITY_BATCH * 2, desc=f"{kind} {severity}")
        p = ensemble_scores(clean, fixed_degradation=(kind, severity), desc=f"{kind} {severity}")[0]
        accepted = sigmoid(q_out[:, 0]) < QUALITY_THRESHOLD
        y = clean.referable.to_numpy()
        sens, spec = M.sens_spec(y, p, DR_THRESHOLD)
        gated = M.sens_spec(y[accepted], p[accepted], DR_THRESHOLD) if accepted.any() else (float("nan"), float("nan"))
        rows.append({"kind": kind, "severity": severity, "rejected": 1 - accepted.mean(), "sens_no_gate": sens, "spec_no_gate": spec, "sens_with_gate": gated[0], "spec_with_gate": gated[1]})
robust = pd.DataFrame(rows).sort_values(["kind", "severity"])
robust.to_csv(REPORT_DIR / "robustness.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
for kind, part in robust.groupby("kind"):
    axes[0].plot(part.severity, part.rejected, marker="o", label=kind)
    axes[1].plot(part.severity, part.sens_no_gate, marker="o", label=kind)
    axes[2].plot(part.severity, part.spec_no_gate, marker="o", label=kind)
axes[0].set(title="Rejected by the quality gate", xlabel="severity", ylim=(-0.03, 1.03))
axes[1].set(title="Sensitivity without the gate", xlabel="severity", ylim=(-0.03, 1.03))
axes[2].set(title="Specificity without the gate", xlabel="severity", ylim=(-0.03, 1.03))
axes[1].axhline(TARGET_SENSITIVITY, ls="--", lw=0.8, color="grey")
axes[2].axhline(TARGET_SPECIFICITY, ls="--", lw=0.8, color="grey")
axes[0].legend(fontsize=8)
show(fig, "robustness_field_conditions")
severe = robust[robust.severity >= 0.7]
print(f"On {len(clean)} good test images: the gate rejects {severe.rejected.mean():.1%} of severely degraded copies (severity >= 0.7) and {robust[(robust.severity > 0) & (robust.severity <= 0.3)].rejected.mean():.1%} of mildly degraded ones (<= 0.3).")
metrics["robustness"] = {"images": int(len(clean)), "rows": robust.to_dict("records"), "severe_rejected": float(severe.rejected.mean())}

## 11. Export

Writes everything the app needs into `fundalux_outputs/`, reloads it from disk through the real inference
pipeline as a self-test and zips it into `/kaggle/working/fundalux_outputs.zip`.

In [ ]:
def jsonable(obj):
    """Plain JSON types only; NaN/inf become null so any JSON parser (MATLAB, JavaScript) can read the file."""
    if isinstance(obj, dict):
        return {str(k): jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, np.ndarray)):
        return [jsonable(v) for v in obj]
    if isinstance(obj, (bool, np.bool_)):
        return bool(obj)
    if isinstance(obj, (int, np.integer)):
        return int(obj)
    if isinstance(obj, (float, np.floating)):
        return float(obj) if math.isfinite(obj) else None
    return obj


for i, m in enumerate(members):
    m["weights"] = f"fundalux_dr_{i + 1}_{m['arch']}.pt"
    save_weights(m["model"], OUT_DIR / m["weights"])
save_weights(quality_model, OUT_DIR / "fundalux_quality.pt")
config = {
    "format": 2,
    "image_size": IMG_SIZE,
    "min_input_side": 32 if SMOKE_TEST else 224,
    "grade_names": GRADE_NAMES,
    "referable_min_grade": REFERABLE_MIN_GRADE,
    "dr": {
        "models": [{"arch": m["arch"], "weights": m["weights"], "input_size": m["size"]} for m in members],
        "combine": "mean of the members' probabilities", "num_grades": 5, "threshold": DR_THRESHOLD,
    },
    "quality": {"arch": QUALITY_ARCH, "weights": "fundalux_quality.pt", "input_size": QUALITY_SIZE, "threshold": QUALITY_THRESHOLD, "hint_ranges": HINT_RANGES},
    "trained_with": {"torch": torch.__version__, "run": RUN, "date": time.strftime("%Y-%m-%d"), "hours": (time.time() - RUN_START) / 3600, "smoke_test": SMOKE_TEST},
}
internal = metrics["dr"]["results"]["Internal test (EyePACS + APTOS)"]
metrics["simulation_inputs"] = {"sensitivity": internal["sensitivity"], "specificity": internal["specificity"], "p_ungradable": internal["rejected_by_gate"]}

# Per-site thresholds: one global threshold does not transfer between cameras, so each evaluation dataset gets
# its own, tuned for the same sensitivity. Datasets with a validation split are tuned on it and measured on their
# test split; the external ones are tuned on all their images and measured by cross-fitting. The app offers these
# as sites and lets a new site calibrate its own.
sites = SiteRegistry(OUT_DIR / "sites.json", model_fingerprint(config), builtin_profiles(index[index.split.isin(["val", "test", "external"])], VAL_SENSITIVITY))
sites.save()
site_table = pd.DataFrame([{"site": p.name, "threshold": p.threshold, "images": p.n_images, "expected sensitivity": p.expected_sensitivity, "expected specificity": p.expected_specificity, "estimated by": p.estimate} for p in sites.profiles])
print(f"Per-site thresholds (global threshold {DR_THRESHOLD:.3f}):")
display(site_table.round(3) if len(site_table) else "none - too few images per site")
metrics["sites"] = {p.name: {"threshold": p.threshold, "expected_sensitivity": p.expected_sensitivity, "expected_specificity": p.expected_specificity, "estimate": p.estimate} for p in sites.profiles}
(OUT_DIR / "config.json").write_text(json.dumps(jsonable(config), indent=2))
(OUT_DIR / "metrics.json").write_text(json.dumps(jsonable(metrics), indent=2))

# Self-test: load the files just written, exactly as the app will, and compare with the models in memory.
bundle = Fundalux.load(OUT_DIR)
photo = read_image(index[index.split == "test"].file.iloc[0])
result = bundle.predict(photo, skip_gate=True)
cropped = crop_fundus(photo, IMG_SIZE)[0]
with torch.no_grad():
    logits = [m["model"].eval()(to_tensor(model_input(cropped, m["size"])).unsqueeze(0).to(DEVICE)) for m in members]
expected = float(ensemble_probabilities(logits)[0][0])
print("Self-test:", json.dumps(jsonable(result.summary()))[:500])
assert abs(result.referable_score - expected) < 1e-3, f"exported models disagree with the trained ones: {result.referable_score} vs {expected}"

# Serving copies for ONNX Runtime, so the web app needs no PyTorch: each network with its feature map and logits,
# plus the head parameters for the closed-form heatmaps. Checked against PyTorch before zipping.
if EXPORT_ONNX:
    try:
        if IN_KAGGLE or IN_COLAB:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxscript"], check=False)
        export_bundle(OUT_DIR, members=[(m["arch"], m["model"].cpu().eval(), m["size"]) for m in members], quality=quality_model.cpu().eval(), quality_size=QUALITY_SIZE)
        served = Fundalux.load(OUT_DIR, backend="onnx").predict(photo, skip_gate=True)
        gap = abs(served.referable_score - result.referable_score), float(np.abs(served.heatmap - result.heatmap).max())
        assert gap[0] < 1e-3 and gap[1] < 1e-2, f"ONNX copies disagree with PyTorch: score {gap[0]:.2e}, heatmap {gap[1]:.2e}"
        print(f"ONNX export checked against PyTorch: score difference {gap[0]:.1e}, heatmap difference {gap[1]:.1e}")
    except Exception as exc:
        print(f"ONNX export failed ({type(exc).__name__}: {exc}). The .pt files are complete; run python -m fundalux.export later.")
        shutil.rmtree(OUT_DIR / "onnx", ignore_errors=True)
        config.pop("onnx", None)
        (OUT_DIR / "config.json").write_text(json.dumps(jsonable(config), indent=2))
    finally:
        for m in members:
            m["model"].to(DEVICE)
        quality_model.to(DEVICE)

archive = shutil.make_archive(str(WORK / "fundalux_outputs"), "zip", OUT_DIR)
print(f"\n{archive}  ({os.path.getsize(archive) / 1e6:.0f} MB)   total run time {(time.time() - RUN_START) / 3600:.2f} h")
for f in sorted(OUT_DIR.rglob("*")):
    if f.is_file():
        print(f"  {str(f.relative_to(OUT_DIR)):45s} {f.stat().st_size / 1e6:8.2f} MB")
if IN_KAGGLE:
    from IPython.display import FileLink

    print("\nDownload fundalux_outputs.zip from the Output section of this notebook (right-hand sidebar while the session is open, or the Output tab of the saved version).")
    display(FileLink(os.path.basename(archive)))
elif IN_COLAB:
    from google.colab import files

    files.download(archive)

## Done

1. Download `fundalux_outputs.zip` from this notebook's Output and unzip it into the project's `models/` folder
   (so that `models/config.json` exists), replacing what is there. The `checkpoints/` and `fundalux/` folders
   in the Output are not needed.
2. Start the app: `streamlit run app/app.py`.
3. Read `metrics.json` and the figures before trusting the models: the success criteria printed in section 8
   are either met or not, per dataset, and the "threshold transfer check" below them shows whether one
   threshold now works across cameras.